# OpenSpec Studio on Colab


## 1 · Settings

In [2]:
#@title Settings { display-mode: "form" }
#@markdown Projects are created and opened in the UI (Projects in its sidebar). Here you only choose where they live, the password and the coding agent.
USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
#@markdown ↑ keep the projects in Drive so they survive runtime resets
UI_PASSWORD = ""  #@param {type:"string"}
#@markdown ↑ empty = random password

#@markdown ### Coding agent
AGENT_TOOLS = "codex"  #@param ["none", "codex", "claude", "gemini", "opencode", "cursor", "github-copilot", "agents", "all"] {allow-input: true}
#@markdown ↑ **codex = OpenAI Codex**, claude = Claude Code, gemini = Gemini CLI, opencode = OpenCode (uses any of the keys below). cursor / github-copilot run in your editor (no key here). Several at once: `codex,claude`.
OPENAI_API_KEY = ""  #@param {type:"string"}
#@markdown ↑ for **OpenAI Codex** (and OpenCode)
ANTHROPIC_API_KEY = ""  #@param {type:"string"}
#@markdown ↑ for **Claude Code** (and OpenCode)
GEMINI_API_KEY = ""  #@param {type:"string"}
#@markdown ↑ for **Gemini CLI** (and OpenCode)
#@markdown Leave a key empty to read it from Colab Secrets (🔑 on the left) under the same name; that keeps it out of the notebook.
AGENT_FULL_ACCESS = True  #@param {type:"boolean"}
#@markdown ↑ let the agent edit files and run commands without asking (this Colab VM is disposable). Off = the agent may only edit files in the project.

import os, secrets, string

# API keys: the form field wins, otherwise Colab Secrets; exported so the agents (and the UI's status check) see them
_form_keys = {"OPENAI_API_KEY": OPENAI_API_KEY, "ANTHROPIC_API_KEY": ANTHROPIC_API_KEY, "GEMINI_API_KEY": GEMINI_API_KEY}
for _name, _value in _form_keys.items():
    if not _value:
        try:
            from google.colab import userdata
            _value = userdata.get(_name) or ""
        except Exception:
            _value = ""
    if _value:
        os.environ[_name] = _value.strip()

AGENTS = ["codex", "claude", "gemini", "opencode", "cursor", "github-copilot"] if AGENT_TOOLS == "all" else          [t.strip() for t in AGENT_TOOLS.split(",") if t.strip() and t.strip() != "none"]
if not UI_PASSWORD:
    UI_PASSWORD = "".join(secrets.choice(string.ascii_letters + string.digits) for _ in range(14))
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/openspec-projects"
else:
    BASE_DIR = "/content/openspec-projects"
PROJECTS_DIR = BASE_DIR                # every project is a folder in here; the UI creates and opens them
APP_DIR = "/content/openspec-studio"   # the app files from step 3 go here
PORT = 7860
os.makedirs(PROJECTS_DIR, exist_ok=True)
os.makedirs(os.path.join(APP_DIR, "static"), exist_ok=True)

import subprocess

def codex_signed_in():
    """Codex only uses a key given to `codex login --with-api-key` (it ignores the OPENAI_API_KEY variable on its own)."""
    try:
        r = subprocess.run(["codex", "login", "status"], text=True, capture_output=True, stdin=subprocess.DEVNULL)
    except FileNotFoundError:
        return False
    return r.returncode == 0 and "not logged" not in (r.stdout + r.stderr).lower()

def sign_in_codex():
    """Sign Codex in from OPENAI_API_KEY when it isn't already; returns whether it is signed in now."""
    if not codex_signed_in() and os.environ.get("OPENAI_API_KEY"):
        subprocess.run(["codex", "login", "--with-api-key"], input=os.environ["OPENAI_API_KEY"], text=True, capture_output=True)
    return codex_signed_in()

print("Projects:", PROJECTS_DIR)
print("Agents :", ", ".join(AGENTS) or "none")
print("Keys   :", ", ".join(k for k in _form_keys if os.environ.get(k)) or "none")

Projects: /content/openspec-projects
Agents : codex
Keys   : OPENAI_API_KEY


## 2 · Install

In [3]:
#@title Install Node.js, OpenSpec, cloudflared, Flask and your coding agent { display-mode: "form" }
import subprocess

def sh(cmd):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode != 0:
        raise RuntimeError(f"Failed: {cmd}\n{(r.stdout + r.stderr)[-1500:]}")
    return r.stdout.strip()

sh("curl -fsSL https://deb.nodesource.com/setup_22.x | bash -")       # OpenSpec needs Node 20+
sh("apt-get install -y -qq nodejs")
sh("npm install -g @fission-ai/openspec@latest --silent")
sh("pip install -q flask waitress")
sh("test -x /usr/local/bin/cloudflared || (wget -q -O /usr/local/bin/cloudflared "
   "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
   "&& chmod +x /usr/local/bin/cloudflared)")
print("node", sh("node -v"), "· openspec", sh("openspec --version"), "·", sh("cloudflared --version").split("\n")[0])

# ---- coding agent CLIs chosen in step 1, signed in with their API keys ----
AGENT_CLI = {  # agent id: (npm package, key it needs)
    "codex": ("@openai/codex", "OPENAI_API_KEY"),
    "claude": ("@anthropic-ai/claude-code", "ANTHROPIC_API_KEY"),
    "gemini": ("@google/gemini-cli", "GEMINI_API_KEY"),
    "opencode": ("opencode-ai", None),
}
missing = []
for agent in AGENTS:
    if agent not in AGENT_CLI:
        print(f"{agent}: runs in your editor; sign in there")
        continue
    package, key = AGENT_CLI[agent]
    sh(f"npm install -g {package}@latest --silent")
    if agent == "codex":
        state = "signed in with OPENAI_API_KEY" if sign_in_codex() else "NOT signed in"
        if not codex_signed_in():
            missing.append("OPENAI_API_KEY")
    elif key:
        state = "key set" if os.environ.get(key) else f"no {key}"
        if not os.environ.get(key):
            missing.append(key)
    else:
        found = [k for k in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GEMINI_API_KEY") if os.environ.get(k)]
        state = "uses " + ", ".join(found) if found else "no key"
        if not found:
            missing.append("OPENAI_API_KEY (or ANTHROPIC_API_KEY / GEMINI_API_KEY)")
    print(f"{agent}: installed · {state}")
if missing:
    raise RuntimeError(f"Your agent can't run without a key: set {', '.join(missing)} in step 1 (or in Colab Secrets under that name, "
                       "with notebook access on), run step 1 again, then run this step again.")

node v22.23.3 · openspec 1.13.2 · cloudflared version 2026.9.3 (built 2026-09-24-16:07 UTC)
codex: installed · signed in with OPENAI_API_KEY


## 3 · App files


### `server.py`: the web server that runs OpenSpec commands

In [ ]:
%%writefile /content/openspec-studio/server.py
import os, re, json, shlex, shutil, secrets, subprocess, functools, threading, time, uuid
from pathlib import Path
from flask import Flask, request, jsonify, session, send_from_directory, abort


if os.environ.get("PROJECT_DIR"):
    BASE = Path(os.environ["PROJECT_DIR"]).resolve().parent
else:
    BASE = Path(os.environ.get("PROJECTS_DIR", os.path.join(os.getcwd(), "openspec-projects"))).resolve()
BASE.mkdir(parents=True, exist_ok=True)
CURRENT_FILE = BASE / ".openspec-studio.json"
AGENT_TOOLS = os.environ.get("AGENT_TOOLS", "")
ROOT = None
PASSWORD = os.environ.get("APP_PASSWORD", "")
STATIC = Path(__file__).parent / "static"
ANSI = re.compile(r"\x1b\[[0-9;?]*[A-Za-z]")

app = Flask(__name__, static_folder=None)
app.secret_key = os.environ.get("APP_SECRET", secrets.token_hex(32))
app.config.update(SESSION_COOKIE_SAMESITE="Lax", SESSION_COOKIE_HTTPONLY=True)


# ---------- projects ----------
def project_names():
    return sorted(d.name for d in BASE.iterdir() if d.is_dir() and not d.name.startswith("."))


def set_project(name):
    """Make a project the current one and remember it for next time (and for the notebook's agent cell)."""
    global ROOT
    ROOT = (BASE / name).resolve() if name else None
    try:
        CURRENT_FILE.write_text(json.dumps({"current": name or "", "projects": str(BASE)}))
    except OSError:
        pass


def _initial_project():
    names = project_names()
    if os.environ.get("PROJECT_DIR"):
        return Path(os.environ["PROJECT_DIR"]).name
    try:
        saved = json.loads(CURRENT_FILE.read_text()).get("current")
        if saved in names:
            return saved
    except (OSError, ValueError):
        pass
    return names[0] if names else None


set_project(_initial_project())


def need_project():
    if ROOT is None or not ROOT.is_dir():
        abort(400, "Create or open a project first (Projects in the sidebar)")


# ---------- helpers ----------
def openspec(args, timeout=180):
    """Run the OpenSpec CLI in the project folder. Returns (exit code, stdout, stderr)."""
    need_project()
    env = {**os.environ, "NO_COLOR": "1", "FORCE_COLOR": "0", "CI": "1", "TERM": "dumb"}
    try:
        r = subprocess.run(["openspec", *args], cwd=ROOT, env=env, text=True, capture_output=True,
                           stdin=subprocess.DEVNULL, timeout=timeout)
        return r.returncode, ANSI.sub("", r.stdout), ANSI.sub("", r.stderr)
    except FileNotFoundError:
        return 127, "", "openspec CLI not found on PATH"
    except subprocess.TimeoutExpired:
        return 124, "", f"Timed out after {timeout}s"


def openspec_json(args):
    """Run an OpenSpec command with --json and parse the result (object or list)."""
    code, out, err = openspec([*args, "--json"])
    m = re.search(r"^[\[{]", out, re.M)
    if not m:
        return None, (out + err).strip()
    try:
        return json.loads(out[m.start():]), None
    except json.JSONDecodeError:
        return None, (out + err).strip()


def project_file(rel_path):
    need_project()
    p = (ROOT / rel_path).resolve()
    if p != ROOT and ROOT not in p.parents:
        abort(400, "Path is outside the project")
    return p


def rel(p):
    return str(Path(p).resolve().relative_to(ROOT))


def change_dir(name):
    need_project()
    if not re.fullmatch(r"[A-Za-z0-9._-]+", name or ""):
        abort(400, "Invalid change name")
    return ROOT / "openspec" / "changes" / name


def md_files(d):
    return [rel(f) for f in sorted(d.rglob("*")) if f.is_file() and f.suffix in (".md", ".yaml", ".yml")]


def need_auth(f):
    @functools.wraps(f)
    def wrapper(*a, **k):
        if PASSWORD and not session.get("ok"):
            return jsonify(error="auth"), 401
        return f(*a, **k)
    return wrapper


@app.errorhandler(400)
def bad_request(e):
    return jsonify(error=getattr(e, "description", str(e))), 400


# ---------- login ----------
@app.post("/api/login")
def login():
    if not PASSWORD or secrets.compare_digest((request.json or {}).get("password", ""), PASSWORD):
        session["ok"] = True
        session.permanent = True
        return jsonify(ok=True)
    return jsonify(error="Wrong password"), 401


@app.post("/api/logout")
def logout():
    session.clear()
    return jsonify(ok=True)


# ---------- project ----------
@app.get("/api/state")
@need_auth
def state():
    data = {"project": ROOT.name if ROOT else None, "path": str(ROOT) if ROOT else None, "projects": project_names(),
            "base": str(BASE), "initialized": bool(ROOT) and (ROOT / "openspec").is_dir(), "version": None,
            "changes": [], "specs": [], "archive": []}
    if not ROOT:
        return jsonify(data)
    code, ver, _ = openspec(["--version"], timeout=30)
    data["version"] = ver.strip() if code == 0 else None
    if not data["initialized"]:
        return jsonify(data)
    listed, _ = openspec_json(["list"])
    data["changes"] = (listed or {}).get("changes", [])
    known = {c["name"] for c in data["changes"]}
    for d in sorted((ROOT / "openspec" / "changes").glob("*/")):
        if d.name != "archive" and d.name not in known:
            data["changes"].append({"name": d.name, "completedTasks": 0, "totalTasks": 0})
    spec_root = ROOT / "openspec" / "specs"
    for f in sorted(spec_root.rglob("spec.md")):
        text = f.read_text(errors="ignore")
        data["specs"].append({"id": str(f.parent.relative_to(spec_root)), "path": rel(f),
                              "requirements": len(re.findall(r"^###\s+Requirement:", text, re.M))})
    arch = ROOT / "openspec" / "changes" / "archive"
    data["archive"] = sorted((d.name for d in arch.glob("*/")), reverse=True) if arch.is_dir() else []
    return jsonify(data)


@app.get("/api/projects")
@need_auth
def projects():
    out = []
    for name in project_names():
        d = BASE / name
        ch = d / "openspec" / "changes"
        remote = subprocess.run(["git", "config", "--get", "remote.origin.url"], cwd=d, text=True, capture_output=True).stdout.strip()
        out.append({"name": name, "current": ROOT is not None and d == ROOT, "initialized": (d / "openspec").is_dir(),
                    "changes": len([c for c in ch.glob("*/") if c.name != "archive"]) if ch.is_dir() else 0,
                    "remote": remote, "modified": int(d.stat().st_mtime)})
    return jsonify(projects=out, base=str(BASE), current=ROOT.name if ROOT else None, tools=AGENT_TOOLS)


@app.post("/api/projects")
@need_auth
def new_project():
    """Create an empty project, or clone a git repository, then set OpenSpec up in it."""
    b = request.json or {}
    git = (b.get("git") or "").strip()
    name = re.sub(r"[^A-Za-z0-9._-]+", "-", (b.get("name") or "").strip()).strip("-.")
    if not name and git:
        name = re.sub(r"\.git$", "", git.rstrip("/").split("/")[-1].split(":")[-1])
    if not name or name.startswith("."):
        abort(400, "Give the project a name, e.g. my-app")
    d = BASE / name
    if d.exists():
        abort(400, f"A project called {name} already exists; open it instead")
    log = []
    if git:
        if not re.fullmatch(r"(https?://|git@|ssh://)[\w.@:/~+-]+", git):
            abort(400, "Enter a git URL such as https://github.com/you/repo.git")
        r = subprocess.run(["git", "clone", git, str(d)], text=True, capture_output=True, timeout=600)
        log.append(f"$ git clone {git}\n{(r.stdout + r.stderr).strip()}")
        if r.returncode != 0:
            shutil.rmtree(d, ignore_errors=True)
            abort(400, f"Clone failed:\n{(r.stdout + r.stderr).strip()[-800:]}")
    else:
        d.mkdir(parents=True)
        subprocess.run(["git", "init", "-q"], cwd=d)
        log.append(f"$ git init  ({d})")
    set_project(name)
    tools = b.get("tools") or AGENT_TOOLS or "none"
    if not (d / "openspec").is_dir() and re.fullmatch(r"[a-z0-9,-]+", tools):
        code, out, err = openspec(["init", "--tools", tools, "--no-animation"])
        log.append(f"$ openspec init --tools {tools}\n{(out + err).strip()}")
    return jsonify(name=name, output="\n\n".join(log))


@app.post("/api/projects/select")
@need_auth
def select_project():
    name = (request.json or {}).get("name", "")
    if name not in project_names():
        abort(400, "No such project")
    set_project(name)
    return jsonify(name=name)


@app.post("/api/init")
@need_auth
def init():
    need_project()
    tools = (request.json or {}).get("tools") or "none"
    if not re.fullmatch(r"[a-z0-9,-]+", tools):
        abort(400, "Invalid tools list")
    if not (ROOT / ".git").exists():
        subprocess.run(["git", "init", "-q"], cwd=ROOT)
    code, out, err = openspec(["init", "--tools", tools, "--no-animation"])
    return jsonify(code=code, output=(out + err).strip())


# ---------- changes ----------
@app.post("/api/changes")
@need_auth
def new_change():
    name = re.sub(r"[^a-z0-9-]+", "-", (request.json or {}).get("name", "").strip().lower()).strip("-")
    if not name:
        abort(400, "Give the change a name, e.g. add-dark-mode")
    code, out, err = openspec(["new", "change", name])
    if code != 0 and not change_dir(name).exists():
        abort(400, (out + err).strip())
    return jsonify(name=name)


@app.get("/api/changes/<name>")
@need_auth
def get_change(name):
    d = change_dir(name)
    if not d.exists():
        abort(400, "Change not found")
    status, err = openspec_json(["status", "--change", name])
    done = total = 0
    if (d / "tasks.md").exists():
        t = (d / "tasks.md").read_text(errors="ignore")
        total = len(re.findall(r"^\s*[-*] \[[ xX]\]", t, re.M))
        done = len(re.findall(r"^\s*[-*] \[[xX]\]", t, re.M))
    apply, _ = openspec_json(["instructions", "apply", "--change", name])
    return jsonify(name=name, dir=rel(d), status=status, statusError=err, files=md_files(d),
                   tasks={"done": done, "total": total}, applyState=(apply or {}).get("state"),
                   skipSpecs=skip_specs(d))


SKIP_RE = re.compile(r"^skip_specs:\s*(\S+)\s*$", re.M)


def skip_specs(d):
    """Does the change's .openspec.yaml declare skip_specs: true?"""
    f = d / ".openspec.yaml"
    m = SKIP_RE.search(f.read_text(errors="ignore")) if f.exists() else None
    return bool(m) and m.group(1).lower() in ("true", "yes", "on")


@app.post("/api/changes/<name>/skip-specs")
@need_auth
def set_skip_specs(name):
    """Mark a change as touching no specs (or undo that). OpenSpec then accepts zero deltas."""
    d = change_dir(name)
    if not d.exists():
        abort(400, "Change not found")
    skip = bool((request.json or {}).get("skip", True))
    f = d / ".openspec.yaml"
    text = SKIP_RE.sub("", f.read_text(errors="ignore")).rstrip() if f.exists() else "schema: spec-driven"
    if skip:
        text += "\nskip_specs: true"
    f.write_text(text.strip() + "\n")
    return jsonify(ok=True, skipSpecs=skip)


@app.post("/api/changes/<name>/scaffold")
@need_auth
def scaffold(name):
    """Create an artifact file from OpenSpec's template for it."""
    b = request.json or {}
    info, err = openspec_json(["instructions", b.get("artifact", ""), "--change", name])
    if not info:
        abort(400, err or "Could not load the template")
    out = info.get("outputPath", "")
    if "*" in out:
        cap = (b.get("capability") or "").strip().strip("/")
        if not re.fullmatch(r"[a-z0-9][a-z0-9/-]*", cap):
            abort(400, "Enter a capability path such as user-auth or ui/theme")
        out = f"specs/{cap}/spec.md"
    target = change_dir(name) / out
    if not target.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(info.get("template") or f"# {b.get('artifact')}\n")
    return jsonify(path=rel(target))


# ---------- files ----------
@app.get("/api/file")
@need_auth
def read_file():
    p = project_file(request.args.get("path", ""))
    if not p.is_file():
        abort(400, "File not found")
    return jsonify(path=rel(p), content=p.read_text(errors="ignore"))


@app.put("/api/file")
@need_auth
def write_file():
    b = request.json or {}
    p = project_file(b.get("path", ""))
    if p.relative_to(ROOT).parts[:1] != ("openspec",):
        abort(400, "Only files inside openspec/ can be edited")
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(b.get("content", ""))
    return jsonify(ok=True)


@app.delete("/api/file")
@need_auth
def delete_file():
    p = project_file(request.args.get("path", ""))
    if p.relative_to(ROOT).parts[:1] != ("openspec",) or not p.is_file():
        abort(400, "Only files inside openspec/ can be deleted")
    p.unlink()
    return jsonify(ok=True)


@app.get("/api/archive/<name>")
@need_auth
def archived(name):
    d = ROOT / "openspec" / "changes" / "archive" / name
    if not re.fullmatch(r"[A-Za-z0-9._-]+", name) or not d.is_dir():
        abort(400, "Not found")
    return jsonify(name=name, files=md_files(d))


# ---------- OpenSpec commands ----------
@app.post("/api/validate")
@need_auth
def validate():
    b = request.json or {}
    args = ["validate", b["name"]] if b.get("name") else ["validate", "--all"]
    data, err = openspec_json(args + ["--strict"])
    return jsonify(result=data, error=err)


@app.post("/api/archive")
@need_auth
def archive():
    name = (request.json or {}).get("name", "")
    change_dir(name)
    code, out, err = openspec(["archive", name, "--yes"])
    return jsonify(code=code, output=(out + err).strip())


@app.post("/api/run")
@need_auth
def run_command():
    """Console: run any non-interactive openspec command (no shell)."""
    try:
        args = shlex.split((request.json or {}).get("cmd", ""))
    except ValueError as e:
        abort(400, str(e))
    if args and args[0] == "openspec":
        args = args[1:]
    if not args:
        abort(400, "Type an openspec command, e.g. list --specs")
    if args[0] in ("view", "completion", "feedback") or (args[0] in ("show", "validate") and len(args) == 1):
        return jsonify(code=1, output="That command is interactive. Pass an item name or use --all.")
    code, out, err = openspec(args)
    return jsonify(code=code, output=(out + err).strip() or "(no output)")



# ---------- coding agents ----------
AGENT_LAYOUTS = [
    ("claude", "Claude Code", ".claude/commands/opsx/{wf}.md", "/opsx:{wf}"),
    ("codex", "OpenAI Codex", ".agents/skills/{skill}/SKILL.md", "{skill}"),
    ("cursor", "Cursor", ".cursor/commands/opsx-{wf}.md", "/opsx-{wf}"),
    ("github-copilot", "GitHub Copilot", ".github/prompts/opsx-{wf}.prompt.md", "/opsx-{wf}"),
    ("gemini", "Gemini CLI", ".gemini/commands/opsx/{wf}.toml", "/opsx:{wf}"),
    ("opencode", "OpenCode", ".opencode/commands/opsx-{wf}.md", "/opsx-{wf}"),
]
SKILLS = {"propose": "openspec-propose", "explore": "openspec-explore", "apply": "openspec-apply-change",
          "archive": "openspec-archive-change", "sync": "openspec-sync-specs", "update": "openspec-update-change",
          "new": "openspec-new-change", "continue": "openspec-continue-change", "ff": "openspec-ff-change",
          "verify": "openspec-verify-change", "bulk-archive": "openspec-bulk-archive-change",
          "onboard": "openspec-onboard"}
ALL_WORKFLOWS = ["propose", "explore", "new", "continue", "ff", "apply", "verify", "update", "sync", "archive",
                 "bulk-archive", "onboard"]
AGENT_DIRS = [".claude", ".agents", ".codex", ".cursor", ".github/prompts", ".gemini", ".opencode", ".windsurf",
              ".devin", ".kiro", ".qwen", ".roo", ".cline", ".continue", ".crush", ".factory", ".amazonq",
              ".augment", ".codebuddy", ".junie", ".kilocode", ".trae", ".zed"]


def agent_auth(tid):
    """Is the agent's CLI installed on this machine, and does it have a key or login?"""
    keys = {"claude": "ANTHROPIC_API_KEY", "gemini": "GEMINI_API_KEY"}
    cli = {"codex": "codex", "claude": "claude", "gemini": "gemini", "opencode": "opencode"}.get(tid)
    if not cli:
        return {"ok": None, "text": "Runs in your editor; sign in there"}
    if not shutil.which(cli):
        return {"ok": False, "text": "CLI not installed here (choose it in the notebook's step 1 and run step 2)"}
    if tid == "codex":
        r = subprocess.run(["codex", "login", "status"], text=True, capture_output=True, stdin=subprocess.DEVNULL)
        text = (r.stdout + r.stderr).strip().splitlines()[-1] if (r.stdout + r.stderr).strip() else "unknown"
        return {"ok": r.returncode == 0 and "not logged" not in text.lower(), "text": text}
    if tid == "opencode":
        found = [k for k in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GEMINI_API_KEY") if os.environ.get(k)]
        return {"ok": bool(found), "text": f"Uses {', '.join(found)}" if found else "No API key set"}
    ok = bool(os.environ.get(keys[tid]))
    return {"ok": ok, "text": f"{keys[tid]} is set" if ok else f"{keys[tid]} is not set"}


def enabled_workflows():
    """The workflows in effect, from "openspec config list".

    Its "Profile settings" block always has an indented comma line such as
    "  workflows: propose, explore, apply (from core profile)". When workflows are set
    explicitly the top of the output also prints them as a YAML list ("workflows:\n  - propose"),
    so only the indented line is read; "config get workflows" is the fallback.
    """
    code, out, _ = openspec(["config", "list"])
    m = re.search(r"^[ \t]+workflows:[ \t]*([a-z][a-z, -]*?)[ \t]*(\(|$)", out, re.M)
    names = [w.strip() for w in m.group(1).split(",") if w.strip()] if m else []
    if not names:
        code, raw, _ = openspec(["config", "get", "workflows"])
        try:
            names = [w for w in json.loads(raw.strip()) if isinstance(w, str)]
        except (json.JSONDecodeError, TypeError):
            names = []
    return [w for w in names if w in ALL_WORKFLOWS]


@app.get("/api/agents")
@need_auth
def agents():
    need_project()
    enabled = enabled_workflows()
    tools = []
    for tid, label, pattern, call in AGENT_LAYOUTS:
        cmds = {}
        for wf in ALL_WORKFLOWS:
            f = ROOT / pattern.format(wf=wf, skill=SKILLS[wf])
            if f.exists():
                cmds[wf] = {"call": call.format(wf=wf, skill=SKILLS[wf]), "file": rel(f)}
        if cmds:
            tools.append({"id": tid, "name": label, "skills": "{skill}" in call, "commands": cmds, "auth": agent_auth(tid)})
    files = []
    for d in AGENT_DIRS:
        base = ROOT / d
        if base.is_dir():
            files += [rel(f) for f in sorted(base.rglob("*"))
                      if f.is_file() and re.search("openspec|opsx", rel(f), re.I)]
    return jsonify(tools=tools, workflows=enabled, allWorkflows=ALL_WORKFLOWS, files=sorted(set(files)), runner=runner_info())


@app.post("/api/agents/add")
@need_auth
def agents_add():
    tools = (request.json or {}).get("tools", "")
    if not re.fullmatch(r"[a-z0-9,-]+", tools):
        abort(400, "Choose an agent")
    code, out, err = openspec(["init", "--tools", tools, "--no-animation"])
    return jsonify(code=code, output=(out + err).strip())


@app.post("/api/agents/workflows")
@need_auth
def agents_workflows():
    """Switch between the core workflows and all of them, then refresh the agent files."""
    preset = (request.json or {}).get("preset")
    if preset == "all":
        steps = [["config", "set", "profile", "custom"], ["config", "set", "workflows", json.dumps(ALL_WORKFLOWS)]]
    else:
        steps = [["config", "set", "profile", "core"], ["config", "unset", "workflows"]]
    log = []
    for args in steps + [["update", "--force"]]:
        code, out, err = openspec(args)
        log.append(f"$ openspec {' '.join(args)}\n{(out + err).strip()}")
    return jsonify(output="\n\n".join(log))


# ---------- run a coding agent from the UI ----------
AGENT_CLIS = {"codex": "OpenAI Codex", "claude": "Claude Code", "gemini": "Gemini CLI", "opencode": "OpenCode"}
FULL_ACCESS = os.environ.get("AGENT_FULL_ACCESS", "1") != "0"
DEFAULT_AGENT = os.environ.get("DEFAULT_AGENT", "")
JOBS, JOBS_LOCK = {}, threading.Lock()
NON_INTERACTIVE = ("You are running non-interactively from a web UI and nobody can answer questions: do not ask any, "
                   "make reasonable assumptions and note them in what you write. Work only inside this project. "
                   "The project already uses OpenSpec (openspec/ exists); never run openspec init.")


def runner_info():
    installed = [{"id": t, "name": n, "auth": agent_auth(t)} for t, n in AGENT_CLIS.items() if shutil.which(t)]
    running = next((j["id"] for j in JOBS.values() if j["status"] == "running"), None)
    default = DEFAULT_AGENT if any(a["id"] == DEFAULT_AGENT for a in installed) else (installed[0]["id"] if installed else "")
    keys = [k for k in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GEMINI_API_KEY") if os.environ.get(k)]
    return {"installed": installed, "default": default, "fullAccess": FULL_ACCESS, "running": running, "keys": keys}


def skill_file(agent, wf):
    """The file where this agent keeps OpenSpec's instructions for a workflow, or None."""
    pattern = next((p for t, _, p, _ in AGENT_LAYOUTS if t == agent), None)
    for pat in ([pattern] if pattern else []) + [".agents/skills/{skill}/SKILL.md"]:
        f = ROOT / pat.format(wf=wf, skill=SKILLS[wf])
        if f.exists():
            return rel(f)
    return None


def agent_command(agent, prompt, model):
    env = {**os.environ, "NO_COLOR": "1", "FORCE_COLOR": "0", "CI": "1", "TERM": "dumb"}
    if agent == "codex":
        cmd = ["codex", "exec", "--skip-git-repo-check", "-C", str(ROOT), *(["-m", model] if model else []),
               *(["--dangerously-bypass-approvals-and-sandbox"] if FULL_ACCESS else ["--sandbox", "workspace-write"]), prompt]
    elif agent == "claude":
        env["IS_SANDBOX"] = "1"
        cmd = ["claude", "-p", prompt, *(["--model", model] if model else []),
               *(["--dangerously-skip-permissions"] if FULL_ACCESS else ["--permission-mode", "acceptEdits"])]
    elif agent == "gemini":
        cmd = ["gemini", "-p", prompt, *(["-m", model] if model else []), *(["--yolo"] if FULL_ACCESS else ["--approval-mode", "auto_edit"])]
    else:
        cmd = ["opencode", "run", *(["-m", model] if model else []), prompt]
    return cmd, env


WF_FALLBACK = {
    "continue": "write the next missing planning artifact: run `openspec status --change {q} --json` to find it and "
                "`openspec instructions <artifact> --change {q} --json` for its template and rules, read the artifacts already written and base it on them",
    "ff": "write every planning artifact the change is still missing, in dependency order (proposal, spec deltas, design, tasks): use "
          "`openspec status --change {q} --json` and `openspec instructions <artifact> --change {q} --json`, base each on the artifacts already written, "
          "then run `openspec validate {q} --strict` and fix anything it reports",
    "verify": "check that the implementation matches the change: read openspec/changes/{c}/ (proposal, specs, design, tasks), inspect the code and tests, "
              "list what is missing, different or untested, and correct the task ticks in tasks.md to reflect reality; do not archive",
}


def build_prompt(agent, task, b):
    """Turn a UI action into a prompt for the agent. Returns (title for the console, prompt)."""
    change, text = b.get("change") or "", (b.get("text") or "").strip()
    existing = [f.split("/", 3)[-1] for f in md_files(change_dir(change)) if f.endswith(".md")] if change else []
    quoted = f'"{change}"'
    if task == "workflow":
        wf = b.get("workflow")
        if wf not in ALL_WORKFLOWS:
            abort(400, "Unknown workflow")
        f = skill_file(agent, wf)
        target = f" for the change {quoted}" if change else ""
        if f:
            return (f"{wf} {change}".strip(), f'Follow the instructions in {f} to run the OpenSpec "{wf}" workflow{target}. {text}'.strip())
        if wf in WF_FALLBACK and change:
            return (f"{wf} {change}", f'Run the OpenSpec "{wf}" workflow{target}: {WF_FALLBACK[wf].format(q=quoted, c=change)}. {text}'.strip())
        abort(400, f"{AGENT_CLIS[agent]} has no '{wf}' instructions in this project: turn on all workflows on the Agents page")
    if task == "plan":
        if not text and not existing:
            abort(400, "Describe what you want first")
        f = skill_file(agent, "ff") or skill_file(agent, "continue") or skill_file(agent, "propose")
        state = "is empty" if not existing else f"already has {', '.join(existing)}: read them and keep them"
        return (f"plan {change}",
                f'The OpenSpec change {quoted} already exists in this project at openspec/changes/{change}/ and {state}. Do not create it again. '
                f'Write every planning file it is still missing, in dependency order (proposal, spec deltas under specs/<capability>/spec.md, design, tasks). '
                f'Run `openspec status --change {quoted} --json` to see which artifact is next and `openspec instructions <artifact> --change {quoted} --json` '
                f'for each file\'s template, rules and instruction; use the template as the structure and never copy the context or rules into the file. '
                f'Inspect the codebase so the plan fits it. ' + (f'The instructions in {f} describe this workflow in detail. ' if f else '') +
                (f'Here is what we want:\n\n{text}\n\n' if text else 'Base everything on the files already in the change. ') +
                f'When every artifact is done, run `openspec validate {quoted} --strict` and fix anything it reports.')
    if task == "write":
        art = b.get("artifact")
        if art not in ("proposal", "specs", "design", "tasks"):
            abort(400, "Unknown artifact")
        if not text and not existing:
            abort(400, "Describe what you want first")
        cap = (b.get("capability") or "").strip().strip("/")
        if cap and not re.fullmatch(r"[a-z0-9][a-z0-9/-]*", cap):
            abort(400, "Enter a capability path such as user-auth or ui/theme")
        where = (f"at specs/{cap}/spec.md" if cap else "under specs/<capability-path>/spec.md, choosing the capability path (reuse an existing spec's path from openspec/specs/ when this modifies it)") \
            if art == "specs" else f"({art}.md)"
        f = skill_file(agent, "continue") or skill_file(agent, "ff")
        return (f"write {art} for {change}",
                f'In the OpenSpec change {quoted} (openspec/changes/{change}/), write the {art} artifact {where}. '
                f'Run `openspec instructions {art} --change {quoted} --json` for its template, rules and the finished files to read for context; '
                f'use the template as the structure and do not copy the context or rules into the file. Inspect the relevant code so it fits the project. '
                + (f'The instructions in {f} describe how artifacts are written. ' if f else '')
                + (f'Base it on this description:\n\n{text}\n\n' if text else f'Base it on the files already in the change ({", ".join(existing)}). ')
                + f'Only edit files under openspec/changes/{change}/. '
                f'When done, run `openspec validate {quoted} --strict` and fix any issue in what you wrote.')
    if task == "fix-validation":
        spec = (b.get("spec") or "").strip().strip("/")
        if spec:
            if not re.fullmatch(r"[a-z0-9][a-z0-9/-]*", spec):
                abort(400, "Invalid spec id")
            return (f"fix validation of spec {spec}",
                    f'Run `openspec validate {spec} --type spec --strict --json` in this project and fix every issue it reports by editing '
                    f'openspec/specs/{spec}/spec.md only, keeping its intent. Re-run validate until it passes, then print its final output.')
        if not change:
            abort(400, "Which change?")
        return (f"fix validation of {change}",
                f'Run `openspec validate {quoted} --strict --json` in this project and fix every issue it reports by editing the files under '
                f'openspec/changes/{change}/ (proposal.md, specs/**/spec.md, design.md, tasks.md). Keep the intent of what is there; replace template '
                f'placeholders with real content grounded in the proposal and the codebase; every requirement uses SHALL or MUST and has at least one '
                f'"#### Scenario:" block. If the change has no spec deltas yet, write the ones the proposal implies under specs/<capability>/spec.md '
                f'(see `openspec instructions specs --change {quoted} --json`); if it truly changes no specs, set skip_specs: true in its .openspec.yaml instead. '
                f'Do not touch anything outside openspec/. Re-run validate until it passes, then print its final output.')
    if task == "custom":
        if not text:
            abort(400, "Type what the agent should do")
        return (text[:70] + ("…" if len(text) > 70 else ""), text)
    abort(400, "Unknown task")


def _reader(job, proc):
    try:
        for line in proc.stdout:
            with JOBS_LOCK:
                job["out"] += ANSI.sub("", line)
    finally:
        code = proc.wait()
        changed = subprocess.run(["git", "status", "--short"], cwd=job["root"], text=True, capture_output=True).stdout.strip()
        with JOBS_LOCK:
            job["code"] = code
            job["status"] = "stopped" if job["status"] == "stopping" else "done"
            job["changed"] = changed
            job["ended"] = time.time()


@app.get("/api/agent")
@need_auth
def agent_info():
    return jsonify(runner_info())


KEY_HINT = {"codex": "set OPENAI_API_KEY", "claude": "set ANTHROPIC_API_KEY", "gemini": "set GEMINI_API_KEY",
            "opencode": "set OPENAI_API_KEY, ANTHROPIC_API_KEY or GEMINI_API_KEY"}


def ensure_signed_in(agent):
    """Codex only uses a key that was passed to `codex login --with-api-key` (the env var alone is ignored),
    so sign it in from the key when needed; for the others the key in the environment is what they use."""
    a = agent_auth(agent)
    if agent == "codex" and not a["ok"] and os.environ.get("OPENAI_API_KEY"):
        subprocess.run(["codex", "login", "--with-api-key"], input=os.environ["OPENAI_API_KEY"], text=True, capture_output=True)
        a = agent_auth(agent)
    if a["ok"] is False:
        abort(400, f"{AGENT_CLIS[agent]} isn't signed in on this machine ({a['text']}). {KEY_HINT[agent]} in the notebook's step 1 "
                   f"(or Colab Secrets), run step 2, then run step 4 again.")


@app.post("/api/agent/run")
@need_auth
def agent_run():
    b = request.json or {}
    agent = b.get("agent") or runner_info()["default"]
    if agent not in AGENT_CLIS or not shutil.which(agent):
        abort(400, "No coding agent CLI is installed here: choose one in the notebook's step 1 and run step 2")
    need_project()
    ensure_signed_in(agent)
    model = (b.get("model") or "").strip()
    if model and not re.fullmatch(r"[A-Za-z0-9._:/-]+", model):
        abort(400, "Invalid model id")
    title, prompt = build_prompt(agent, b.get("task"), b)
    prompt = f"{prompt}\n\n{NON_INTERACTIVE}"
    with JOBS_LOCK:
        if any(j["status"] in ("running", "stopping") for j in JOBS.values()):
            return jsonify(error="An agent is already running; stop it or wait for it to finish"), 409
        cmd, env = agent_command(agent, prompt, model)
        try:
            proc = subprocess.Popen(cmd, cwd=ROOT, env=env, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                                    stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
        except OSError as e:
            abort(400, f"Could not start {agent}: {e}")
        job = {"id": uuid.uuid4().hex[:10], "agent": agent, "title": f"{AGENT_CLIS[agent]} · {title}", "prompt": prompt, "root": str(ROOT),
               "out": "", "code": None, "status": "running", "changed": "", "started": time.time(), "proc": proc}
        JOBS[job["id"]] = job
        for old in [j for j in JOBS.values() if j["status"] not in ("running", "stopping")][:-20]:
            JOBS.pop(old["id"], None)
    threading.Thread(target=_reader, args=(job, proc), daemon=True).start()
    return jsonify(id=job["id"], title=job["title"], agent=agent, prompt=prompt)


@app.get("/api/agent/jobs/<job_id>")
@need_auth
def agent_job(job_id):
    job = JOBS.get(job_id)
    if not job:
        abort(400, "No such run")
    offset = max(0, int(request.args.get("offset", 0) or 0))
    with JOBS_LOCK:
        out = job["out"]
        return jsonify(id=job_id, title=job["title"], status=job["status"], code=job["code"], output=out[offset:], length=len(out),
                       changed=job["changed"], seconds=int((job.get("ended") or time.time()) - job["started"]))


@app.post("/api/agent/jobs/<job_id>/stop")
@need_auth
def agent_stop(job_id):
    job = JOBS.get(job_id)
    if not job:
        abort(400, "No such run")
    with JOBS_LOCK:
        if job["status"] == "running":
            job["status"] = "stopping"
            try:
                os.killpg(os.getpgid(job["proc"].pid), 15)
            except (ProcessLookupError, PermissionError):
                pass
    return jsonify(ok=True)


# ---------- page ----------
@app.get("/")
def index():
    return send_from_directory(STATIC, "index.html")


@app.get("/static/<path:name>")
def static_files(name):
    return send_from_directory(STATIC, name)


if __name__ == "__main__":
    port = int(os.environ.get("PORT", 7860))
    try:
        from waitress import serve
        serve(app, host="127.0.0.1", port=port, threads=8)
    except ImportError:
        app.run(host="127.0.0.1", port=port, threaded=True)


### `static/index.html`: the page

In [5]:
%%writefile /content/openspec-studio/static/index.html
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>OpenSpec Studio</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Public+Sans:wght@400;500;600;700&family=Source+Serif+4:opsz,wght@8..60,400;8..60,600&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
<script src="https://cdnjs.cloudflare.com/ajax/libs/marked/12.0.2/marked.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/dompurify/3.1.6/purify.min.js"></script>
<link rel="stylesheet" href="/static/app.css">
</head>
<body>
<div id="root"></div>
<div class="toast" id="toast"></div>

<dialog id="dlg"><div class="card" id="dlgBody"></div></dialog>

<script src="/static/app.js"></script>
</body>
</html>


Writing /content/openspec-studio/static/index.html


### `static/app.css`: the styles

In [6]:
%%writefile /content/openspec-studio/static/app.css
:root{
  color-scheme:dark;
  --paper:#0E131A; --sheet:#151C26; --sheet-2:#1B2330; --ink:#E3E9F1; --ink-2:#9AA7B8; --rule:#263142;
  --side:#0A0E14; --side-ink:#D5DEE9; --side-dim:#7D8BA0; --side-hi:#18212D;
  --blue:#7AA2F7; --blue-strong:#3D6FD9; --blue-soft:#18253D; --green:#5CCB95; --green-soft:#12291F;
  --amber:#E5AE55; --amber-soft:#2E2414; --red:#F2877D; --red-soft:#35191A; --grey:#5E6B7E;
  --sans:"Public Sans",system-ui,-apple-system,"Segoe UI",sans-serif;
  --serif:"Source Serif 4",Georgia,serif; --mono:"JetBrains Mono",ui-monospace,Menlo,monospace;
  --dock:380px;
}
*{box-sizing:border-box}
html,body,#root{height:100%;margin:0}
body{font:15px/1.5 var(--sans);color:var(--ink);background:var(--paper)}
button,input,textarea,select{font:inherit;color:inherit}
:focus-visible{outline:2px solid var(--blue);outline-offset:2px}
.app{display:grid;grid-template-columns:290px minmax(0,1fr) var(--dock);height:100%}
.app.dock-closed{grid-template-columns:290px minmax(0,1fr) 0}
.app.dock-closed .dock{display:none}
@media (min-width:821px) and (max-width:1200px){:root{--dock:330px}.app{grid-template-columns:240px minmax(0,1fr) var(--dock)}.app.dock-closed{grid-template-columns:240px minmax(0,1fr) 0}}
.dock{position:relative;display:flex;flex-direction:column;min-height:0;min-width:0;background:#080B10;border-left:1px solid var(--rule)}
.dock-grip{position:absolute;left:-3px;top:0;bottom:0;width:7px;cursor:col-resize;touch-action:none;z-index:2}
.dock-grip:hover,.dock-grip:active{background:var(--blue-strong);opacity:.7}
.dock-head{display:flex;align-items:center;gap:8px;padding:12px 14px;border-bottom:1px solid var(--rule)}
.dock-head b{font-size:14px}
.dock-head .muted{flex:1;font:12.5px var(--mono);white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.dock .chips{padding:10px 12px 0;margin:0}
.dock .console{flex:1;min-height:0;max-height:none;border:0;border-radius:0;margin:0;padding:12px 14px;font-size:12.5px}
.dock .cmdline{margin:0;padding:10px 12px;border-top:1px solid var(--rule)}
.dock .cmdline input{min-width:0;flex:1}
.dock .cmdhelp{margin:0;padding:0 12px 10px;font-size:12px;line-height:1.8}
aside{background:var(--side);color:var(--side-ink);display:flex;flex-direction:column;min-height:0}
.brand{padding:20px 20px 14px;border-bottom:1px solid var(--side-hi)}
.brand h1{margin:0;font-size:17px;font-weight:700;letter-spacing:-.01em}
.brand p{margin:2px 0 0;font-size:13px;color:var(--side-dim);word-break:break-all}
.nav{flex:1;overflow:auto;padding:10px 10px 20px}
.nav h2{font-size:12.5px;font-weight:600;color:var(--side-dim);margin:18px 10px 6px;display:flex;justify-content:space-between}
.item{display:block;width:100%;text-align:left;background:none;border:0;border-radius:6px;padding:8px 10px;cursor:pointer;color:var(--side-ink)}
.item:hover{background:var(--side-hi)}
.item.on{background:var(--side-hi);box-shadow:inset 3px 0 0 var(--blue)}
.item small{display:block;color:var(--side-dim);font-size:12px}
.bar{height:3px;background:#243044;border-radius:2px;margin-top:6px;overflow:hidden}
.bar i{display:block;height:100%;background:var(--green)}
.empty{color:var(--side-dim);font-size:13px;padding:4px 10px}
.side-foot{padding:12px;border-top:1px solid var(--side-hi);display:flex;gap:6px}
.side-foot .item{text-align:center;font-size:13px;padding:8px 4px}
main{min-width:0;overflow:auto;container-type:inline-size}
.wrap{max-width:1000px;margin:0 auto;padding:28px 32px 60px}
.head{display:flex;flex-wrap:wrap;align-items:flex-end;gap:12px;justify-content:space-between;margin-bottom:22px}
.head h2{margin:0;font-size:26px;letter-spacing:-.015em;font-weight:700}
.head p{margin:4px 0 0;color:var(--ink-2)}
.btn{border:1px solid var(--rule);background:var(--sheet);border-radius:7px;padding:7px 13px;cursor:pointer;font-weight:500;font-size:14px}
.btn:hover{border-color:var(--ink-2)}
.btn.primary{background:var(--blue-strong);border-color:var(--blue-strong);color:#fff}
.btn.primary:hover{background:#4C7DE6;border-color:#4C7DE6}
.btn.danger{color:var(--red)}
.btn:disabled{opacity:.5;cursor:default}
.row{display:flex;gap:8px;flex-wrap:wrap;align-items:center}
.pipe{display:grid;grid-template-columns:repeat(auto-fit,minmax(108px,1fr));gap:1px;margin:0 0 26px;background:var(--sheet);border:1px solid var(--rule);border-radius:10px;overflow:hidden}
.stage{padding:14px 14px;border:0;position:relative;cursor:pointer;background:var(--sheet);text-align:left;box-shadow:0 0 0 1px var(--rule)}
.stage:hover{background:var(--paper)}
.stage.on{background:var(--blue-soft)}
.stage b{display:block;font-size:15px;text-transform:capitalize;padding-right:16px}
.stage span{display:block;font-size:12.5px;color:var(--ink-2)}
.stage .dot{position:absolute;top:14px;right:14px;width:11px;height:11px;border-radius:50%;border:2px solid var(--grey)}
.stage.done .dot{background:var(--green);border-color:var(--green)}
.stage.ready .dot{border-color:var(--blue);background:var(--blue-soft)}
.stage.blocked{opacity:.62}
.panel{background:var(--sheet);border:1px solid var(--rule);border-radius:10px}
.tabs{display:flex;gap:2px;overflow-x:auto;border-bottom:1px solid var(--rule);padding:6px 8px 0}
.tab{border:0;background:none;padding:8px 12px;cursor:pointer;border-bottom:2px solid transparent;font-size:13.5px;white-space:nowrap;color:var(--ink-2)}
.tab.on{border-color:var(--blue);color:var(--ink);font-weight:600}
.toolbar{display:flex;justify-content:space-between;align-items:center;gap:8px;padding:10px 14px;border-bottom:1px solid var(--rule);flex-wrap:wrap}
.toolbar code{font:12.5px var(--mono);color:var(--ink-2)}
.doc{padding:22px 30px 30px;font-family:var(--serif);font-size:16.5px;line-height:1.65;max-width:78ch}
.doc h1,.doc h2,.doc h3,.doc h4{font-family:var(--sans);line-height:1.25;margin:1.4em 0 .5em}
.doc h1{font-size:24px;margin-top:0}.doc h2{font-size:19px}.doc h3{font-size:16.5px}.doc h4{font-size:15px;color:var(--ink-2)}
.doc code{font:13.5px var(--mono);background:var(--paper);padding:1px 5px;border-radius:4px}
.doc pre{background:var(--paper);padding:12px;border-radius:7px;overflow:auto}
.doc pre code{background:none;padding:0}
.doc ul.contains-task-list{list-style:none;padding-left:4px}
.doc input[type=checkbox]{width:16px;height:16px;margin-right:8px;vertical-align:-2px;cursor:pointer;accent-color:var(--green)}
.doc table{border-collapse:collapse}.doc td,.doc th{border:1px solid var(--rule);padding:4px 8px}
textarea.editor{width:100%;min-height:62vh;border:0;padding:18px 22px;font:14px/1.6 var(--mono);resize:vertical;background:var(--sheet);outline:none}
.note{padding:14px 16px;border-radius:8px;margin:0 0 18px;font-size:14px}
.note.ok{background:var(--green-soft);color:var(--green)}
.note.warn{background:var(--amber-soft);color:var(--amber)}
.note.err{background:var(--red-soft);color:var(--red)}
.note ul{margin:6px 0 0;padding-left:18px}
.card{padding:22px 24px}
.card h3{margin:0 0 6px;font-size:17px}
.card p{margin:0 0 14px;color:var(--ink-2)}
label{display:block;font-weight:600;font-size:13.5px;margin:14px 0 5px}
input[type=text],input[type=password],textarea.field,select{width:100%;border:1px solid var(--rule);background:var(--paper);border-radius:7px;padding:9px 11px}
textarea.field{min-height:110px;resize:vertical}
.hint{font-size:12.5px;color:var(--ink-2);margin-top:4px}
.idea{background:var(--paper);border-left:3px solid var(--blue);padding:10px 14px;border-radius:0 7px 7px 0;margin:0 0 20px;color:var(--ink-2);font-size:14.5px;white-space:pre-wrap}
.console{background:#080B10;border:1px solid var(--rule);color:#C9D3E0;font:13px/1.55 var(--mono);border-radius:10px;padding:16px;min-height:360px;max-height:65vh;overflow:auto;white-space:pre-wrap}
.console .c{color:var(--blue)}
.console .e{color:var(--red)}
.cmdline{display:flex;gap:8px;margin-top:10px}
.cmdline input{font-family:var(--mono)}
.chips{display:flex;gap:6px;flex-wrap:wrap;margin:0 0 12px}
.chip{border:1px solid var(--rule);background:var(--sheet);border-radius:99px;padding:3px 11px;font:12.5px var(--mono);cursor:pointer}
.spec-list{display:grid;gap:8px}
.muted{color:var(--ink-2)}
dialog{border:1px solid var(--rule);border-radius:12px;padding:0;width:min(520px,94vw);background:var(--sheet);color:var(--ink)}
dialog::backdrop{background:rgba(0,0,0,.6)}
dialog .card{padding:22px 24px}
.toast{position:fixed;bottom:20px;right:20px;background:var(--sheet-2);color:var(--ink);border:1px solid var(--rule);padding:10px 16px;border-radius:8px;font-size:14px;opacity:0;transform:translateY(8px);transition:.2s;pointer-events:none;max-width:420px}
.toast.show{opacity:1;transform:none}
.login{display:grid;place-items:center;height:100%}
.login form{width:min(360px,90vw)}
.spin{display:inline-block;width:14px;height:14px;border:2px solid currentColor;border-right-color:transparent;border-radius:50%;animation:s .7s linear infinite;vertical-align:-2px}
@keyframes s{to{transform:rotate(360deg)}}
.menu-btn,.menu-bar{display:none}

@media (max-width:820px){
  .app,.app.dock-closed{grid-template-columns:1fr}
  aside{position:fixed;inset:0 20% 0 0;z-index:5;transform:translateX(-100%);transition:.2s}
  aside.open{transform:none}
  .dock,.app.dock-closed .dock{display:flex;position:fixed;inset:0 0 0 10%;z-index:6;transform:translateX(100%);transition:.2s;box-shadow:-12px 0 30px rgba(0,0,0,.5)}
  .app:not(.dock-closed) .dock{transform:none}
  .dock-grip{display:none}
  .menu-bar{display:flex;gap:8px;margin-bottom:12px}
  .menu-btn{display:inline-block}
  .wrap{padding:18px 16px 50px}
  .pipe{grid-template-columns:1fr}
  .doc{padding:18px}
}
@media (prefers-reduced-motion:reduce){*{transition:none!important;animation:none!important}}
.item.add{color:var(--blue)}
.err-text{color:var(--red)}
.pre{white-space:pre-wrap;margin:8px 0 0;font:13px/1.5 var(--mono)}
.doc.flush{padding:0;font-size:15.5px}
.btn:hover{border-color:var(--ink-2);background:var(--sheet-2)}
input[type=text],input[type=password],select,textarea.editor{background:var(--paper);color:var(--ink);border-color:var(--rule)}
textarea.editor{background:#0B1017}
.doc a{color:var(--blue)}
.doc pre,.doc code{background:#0B1017}
.stage.done .dot{box-shadow:0 0 0 3px var(--green-soft)}
::placeholder{color:#5E6B7E}
::-webkit-scrollbar{width:10px;height:10px}::-webkit-scrollbar-thumb{background:#263142;border-radius:8px}::-webkit-scrollbar-track{background:transparent}
.cmdhelp{font-size:13px;color:var(--ink-2);margin:10px 0 0}
.cmdhelp code{background:var(--sheet);border:1px solid var(--rule);border-radius:4px;padding:0 5px}
.list{overflow:hidden}
.lrow{display:grid;grid-template-columns:1fr auto 140px;gap:16px;align-items:center;width:100%;text-align:left;padding:14px 18px;background:none;border:0;border-bottom:1px solid var(--rule);cursor:pointer}
.lrow:last-child{border-bottom:0}
.lrow:hover{background:var(--sheet-2)}
.meter{height:6px;background:#243044;border-radius:4px;overflow:hidden}
.meter i{display:block;height:100%;background:var(--green)}
.seg{display:inline-flex;border:1px solid var(--rule);border-radius:7px;overflow:hidden}
.seg button{background:none;border:0;padding:6px 12px;font-size:13.5px;color:var(--ink-2);cursor:pointer}
.seg button+button{border-left:1px solid var(--rule)}
.seg button.on{background:var(--blue-soft);color:var(--ink);font-weight:600}
.src{margin:0;padding:20px 26px;font:13.5px/1.65 var(--mono);white-space:pre-wrap;word-break:break-word;color:#C9D3E0;background:#0B1017}
.split{display:grid;grid-template-columns:1fr 1fr;min-height:62vh}
.split>*{min-width:0}
.split .pane{overflow:auto;max-height:75vh}
.split .pane+.pane,.split .editor+.pane{border-left:1px solid var(--rule)}
.split .editor{min-height:62vh;height:100%}
.split .doc{padding:18px 24px}
@media (max-width:820px){.split{grid-template-columns:1fr}.split .pane+.pane,.split .editor+.pane{border-left:0;border-top:1px solid var(--rule)}}
.stage.wf{background:repeating-linear-gradient(135deg,transparent 0 8px,rgba(122,162,247,.035) 8px 16px) var(--sheet)}
.stage.wf.done{background:var(--green-soft)}
.nextbox{background:var(--sheet);border:1px solid var(--rule);border-left:3px solid var(--blue);border-radius:10px;padding:16px 20px;margin:0 0 22px}
.nextbox h3{margin:0 0 4px;font-size:16px}
.nextbox p{margin:0 0 10px}
.wfname{font-weight:600;margin:12px 0 6px}
.cmdrow{display:grid;grid-template-columns:140px 1fr auto;gap:10px;align-items:center;padding:6px 0}
.cmdrow .who{color:var(--ink-2);font-size:13.5px}
.cmdrow code{font:13.5px var(--mono);background:#0B1017;border:1px solid var(--rule);border-radius:6px;padding:6px 10px;overflow-x:auto;white-space:nowrap}
.btn.sm{padding:4px 10px;font-size:13px}
.agentcard,.panel.card{margin-bottom:18px}
.wftable{display:grid;gap:2px}
.wfrow{display:grid;grid-template-columns:auto 130px 1fr auto;gap:12px;align-items:center;padding:7px 0;border-bottom:1px solid var(--rule)}
.agentcard .wfrow{grid-template-columns:230px 1fr auto}
.wfrow:last-child{border-bottom:0}
.wfrow code{font:13px var(--mono);color:var(--blue)}
.wfrow span{color:var(--ink-2);font-size:14px}
.tick{width:22px;height:22px;border-radius:50%;display:grid;place-items:center;font-size:12px;background:var(--sheet-2);color:var(--grey)}
.tick.on{background:var(--green-soft);color:var(--green)}
@media (max-width:820px){.cmdrow{grid-template-columns:1fr}.wfrow,.agentcard .wfrow{grid-template-columns:1fr}}
.stage{display:flex;flex-direction:column;align-items:flex-start;justify-content:flex-start}
.authtag{font-size:12px;font-weight:600;border-radius:99px;padding:2px 10px;margin-left:10px;vertical-align:2px;background:var(--sheet-2);color:var(--ink-2)}
.authtag.ok{background:var(--green-soft);color:var(--green)}
.authtag.bad{background:var(--amber-soft);color:var(--amber)}

.stage.skipped .dot{border-style:dashed;border-color:var(--ink-2)}
.stage.skipped{opacity:.85}
.btn.sm.primary{padding:4px 12px}
@container (max-width:640px){.cmdrow{grid-template-columns:1fr}.wfrow,.agentcard .wfrow{grid-template-columns:1fr}.wrap{padding-left:20px;padding-right:20px}.head h2{font-size:22px}}
.btn.agent{border-color:#2F5E4B;background:#12291F;color:var(--green)}
.btn.agent:hover{background:#173627;border-color:var(--green)}
.btn.agent.primary,.btn.primary.agent{background:#1F7A55;border-color:#1F7A55;color:#fff}
.btn.agent.primary:hover,.btn.primary.agent:hover{background:#25905F;border-color:#25905F}
.console .a{color:var(--green);font-weight:600}
.console .a .btn{margin-left:10px;vertical-align:1px;font-weight:500}
.console .spin{width:11px;height:11px;margin-left:4px;opacity:.7}
.cmdrow .row{gap:6px;flex-wrap:nowrap}
.projsel{display:flex;gap:6px;margin-top:8px}
.projsel select{flex:1;min-width:0;padding:6px 8px;font-size:13.5px;font-weight:600;background:var(--side-hi);border-color:#243044;color:var(--side-ink)}
.projbtn{width:34px;padding:6px 0;text-align:center;flex:none;font-size:15px}
.prow{grid-template-columns:1fr auto auto;cursor:default}
.prow small{display:block;font-size:12.5px;margin-top:2px}
.prow.cur{background:var(--blue-soft)}


Writing /content/openspec-studio/static/app.css


### `static/app.js`: the UI logic

In [7]:
%%writefile /content/openspec-studio/static/app.js

const $ = s => document.querySelector(s);
const esc = s => String(s ?? "").replace(/[&<>"']/g, c => ({"&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;", "'": "&#39;"}[c]));
const md = s => DOMPurify.sanitize(marked.parse(s || "", {gfm: true}));
const S = {state: null, view: {kind: "home"}, change: null, file: null, content: "", editing: false, dirty: false, log: [], dock: true};
const STAGE_TEXT = {proposal: "Why and what", specs: "Requirement deltas", design: "How it's built", tasks: "Implementation checklist"};

/* ---------- utilities ---------- */
async function api(path, opts = {}) {
  const r = await fetch(path, {headers: {"Content-Type": "application/json"}, credentials: "same-origin",
    ...opts, body: opts.body ? JSON.stringify(opts.body) : undefined});
  if (r.status === 401 && path !== "/api/login") { renderLogin(); throw new Error("Sign in first"); }
  const data = await r.json().catch(() => ({}));
  if (!r.ok) throw new Error(data.error || `Request failed (${r.status})`);
  return data;
}
function toast(msg) { const t = $("#toast"); t.textContent = msg; t.classList.add("show"); clearTimeout(t._h); t._h = setTimeout(() => t.classList.remove("show"), 3200); }
function busy(btn, on) { if (!btn) return; if (on) { btn._t = btn.innerHTML; btn.innerHTML = '<span class="spin"></span>'; btn.disabled = true; } else { btn.innerHTML = btn._t; btn.disabled = false; } }
function guard() { return !S.dirty || confirm("You have unsaved edits. Discard them?"); }
function plural(n, word) { return `${n} ${word}${n === 1 ? "" : "s"}`; }
function menuBtn() { return `<div class="menu-bar"><button class="btn menu-btn" onclick="document.getElementById('side').classList.add('open')">☰ Menu</button><button class="btn menu-btn" onclick="dockOpen(true)">&gt;_ Console</button></div>`; }
function dialog(html) { $("#dlgBody").innerHTML = html; $("#dlg").showModal(); const c = $("#cl"); if (c) c.onclick = () => $("#dlg").close(); }

/* ---------- login ---------- */
function renderLogin() {
  $("#root").innerHTML = `<div class="login"><form class="panel card" id="lf">
    <h3>OpenSpec Studio</h3><p>Enter the password printed in your Colab notebook.</p>
    <label for="pw">Password</label><input type="password" id="pw" autofocus>
    <div class="row" style="margin-top:16px"><button class="btn primary">Sign in</button><span id="lerr" class="err-text"></span></div>
  </form></div>`;
  $("#lf").onsubmit = async e => {
    e.preventDefault();
    try { await api("/api/login", {method: "POST", body: {password: $("#pw").value}}); boot(); }
    catch (err) { $("#lerr").textContent = err.message; }
  };
}

/* ---------- layout & navigation ---------- */
async function boot() {
  try { S.state = await api("/api/state"); } catch { return; }
  $("#root").innerHTML = `<div class="app"><aside id="side"></aside><main><div class="wrap" id="main"></div></main>
    <section class="dock" id="dock" aria-label="Console"></section></div>`;
  let open = true, width = "";
  try { open = localStorage.getItem("os.dock") !== "0"; width = localStorage.getItem("os.dockw") || ""; } catch {}
  if (width) document.documentElement.style.setProperty("--dock", width);
  if (!S.state.project) S.view = {kind: "projects"};
  renderSide(); renderDock(); dockOpen(open && window.innerWidth > 820); route();
  if (S.state.project) getAgents().then(() => { updateDockHints(); }).catch(() => {});
}
async function refresh() { S.state = await api("/api/state"); renderSide(); updateDockHints(); }
function go(kind, name) { S.view = {kind, name}; S.dirty = false; S.editing = false; renderSide(); route(); }

function renderSide() {
  const st = S.state, v = S.view;
  const on = (k, n) => v.kind === k && v.name === n ? " on" : "";
  const changes = st.changes.map(c => {
    const pct = c.totalTasks ? Math.round(100 * c.completedTasks / c.totalTasks) : 0;
    return `<button class="item${on("change", c.name)}" data-go="change" data-name="${esc(c.name)}">${esc(c.name)}
      <small>${c.totalTasks ? `${c.completedTasks} of ${c.totalTasks} tasks` : "No tasks yet"}</small>
      <div class="bar"><i style="width:${pct}%"></i></div></button>`;
  }).join("") || `<div class="empty">No active changes</div>`;
  const specs = st.specs.map(s => `<button class="item${on("spec", s.id)}" data-go="spec" data-name="${esc(s.id)}">${esc(s.id)}
      <small>${plural(s.requirements, "requirement")}</small></button>`).join("") || `<div class="empty">Specs appear here after you archive a change</div>`;
  const arch = st.archive.map(a => `<button class="item${on("archived", a)}" data-go="archived" data-name="${esc(a)}">${esc(a)}</button>`).join("")
    || `<div class="empty">Nothing archived yet</div>`;
  const projOpts = st.projects.map(p => `<option value="${esc(p)}"${p === st.project ? " selected" : ""}>${esc(p)}</option>`).join("");
  $("#side").innerHTML = `
    <div class="brand"><h1>OpenSpec Studio</h1>
      <div class="projsel"><select id="proj" title="Switch project">${projOpts || `<option value="">No project yet</option>`}<option value="__new">＋ New project…</option></select>
        <button class="item projbtn${v.kind === "projects" ? " on" : ""}" data-go="projects" title="All projects">⋯</button></div>
      <p>${st.version ? `OpenSpec v${esc(st.version)}` : st.project ? "" : "Create or open a project"}</p></div>
    <div class="nav">
      ${st.project ? `<button class="item${v.kind === "home" ? " on" : ""}" data-go="home">Overview</button>` : ""}
      ${st.initialized ? `
      <h2><span>Changes</span><span>${st.changes.length}</span></h2>
      <button class="item add" data-go="new">+ New change</button>${changes}
      <h2><span>Specs</span><span>${st.specs.length}</span></h2>${specs}
      <h2><span>Archive</span><span>${st.archive.length}</span></h2>${arch}` : ""}
    </div>
    <div class="side-foot">
      <button class="item${S.dock ? " on" : ""}" id="dockToggle" title="Show or hide the console">Console</button>
      ${st.initialized ? `<button class="item${v.kind === "agents" ? " on" : ""}" data-go="agents">Agents</button>` : ""}
      <button class="item" id="logout">Sign out</button>
    </div>`;
  $("#proj").onchange = e => {
    const v = e.target.value;
    if (v === "__new") { e.target.value = st.project || ""; if (guard()) go("projects"); return; }
    if (v && v !== st.project && guard()) openProject(v);
  };
  $("#side").querySelectorAll("[data-go]").forEach(b => b.onclick = () => {
    if (!guard()) return;
    $("#side").classList.remove("open");
    go(b.dataset.go, b.dataset.name);
  });
  $("#dockToggle").onclick = () => { $("#side").classList.remove("open"); dockOpen(!S.dock); };
  $("#logout").onclick = async () => { await api("/api/logout", {method: "POST"}); renderLogin(); };
}

function route() {
  if (!S.state.project && S.view.kind !== "projects") S.view = {kind: "projects"};
  ({home: viewHome, new: viewNew, change: viewChange, spec: viewSpec, archived: viewArchived, agents: viewAgents, projects: viewProjects}[S.view.kind] || viewHome)();
}

/* ---------- projects ---------- */
async function openProject(name) {
  try {
    await api("/api/projects/select", {method: "POST", body: {name}});
    S.agents = null; S.change = null; S.file = null; S.dirty = false; S.editing = false;
    await refresh(); toast(`Opened ${name}`); go("home");
    getAgents().then(updateDockHints).catch(() => {});
  } catch (err) { toast(err.message); }
}

async function viewProjects() {
  let r;
  try { r = await api("/api/projects"); } catch (err) { $("#main").innerHTML = `<div class="note err">${esc(err.message)}</div>`; return; }
  const when = t => { const d = (Date.now() / 1000 - t) / 86400; return d < 1 ? "today" : d < 2 ? "yesterday" : `${Math.round(d)} days ago`; };
  $("#main").innerHTML = `${menuBtn()}<div class="head"><div><h2>Projects</h2><p>Each project is a folder in <code>${esc(r.base)}</code> with its own code, <code>openspec/</code> and git history.</p></div></div>
    ${r.projects.length ? `<div class="panel list" style="margin-bottom:18px">${r.projects.map(p => `<div class="lrow prow${p.current ? " cur" : ""}">
      <div><b>${esc(p.name)}</b>${p.current ? ` <span class="authtag ok">open</span>` : ""}<small class="muted">${p.remote ? esc(p.remote) + " · " : ""}${p.initialized ? plural(p.changes, "active change") : "OpenSpec not set up yet"} · ${when(p.modified)}</small></div>
      <span></span><span class="row" style="justify-content:flex-end">${p.current ? `<button class="btn sm" onclick="go('home')">Overview</button>` : `<button class="btn sm primary" data-open="${esc(p.name)}">Open</button>`}</span></div>`).join("")}</div>`
      : `<div class="note warn">No projects yet. Create one below: a fresh folder, or a clone of a repository you already have.</div>`}
    <div class="panel card"><h3>New project</h3>
      <p>Creates the folder, runs <code>git init</code> (or clones your repo) and sets OpenSpec up in it${r.tools && r.tools !== "none" ? ` for <b>${esc(r.tools)}</b> (the agent chosen in the notebook)` : ""}.</p>
      <label for="pn">Project name</label><input type="text" id="pn" placeholder="my-app">
      <label for="pg">Git repository to clone <span class="muted">(optional)</span></label><input type="text" id="pg" placeholder="https://github.com/you/repo.git">
      <div class="hint">Leave empty for a new empty project. Private repos need a token in the URL, e.g. https://TOKEN@github.com/you/repo.git</div>
      <div class="row" style="margin-top:18px"><button class="btn primary" id="pc">Create project</button></div>
      <pre id="pout" class="pre muted"></pre></div>`;
  $("#main").querySelectorAll("[data-open]").forEach(b => b.onclick = () => { if (guard()) openProject(b.dataset.open); });
  const create = async () => {
    busy($("#pc"), true);
    try {
      const x = await api("/api/projects", {method: "POST", body: {name: $("#pn").value, git: $("#pg").value}});
      $("#pout").textContent = x.output || "";
      S.agents = null; S.change = null; S.file = null;
      await refresh(); toast(`Created ${x.name}`); go("home");
      getAgents().then(updateDockHints).catch(() => {});
    } catch (err) { toast(err.message); $("#pout").textContent = err.message; } finally { busy($("#pc"), false); }
  };
  $("#pc").onclick = create;
  $("#pn").onkeydown = $("#pg").onkeydown = e => { if (e.key === "Enter") create(); };
  ($("#pn").value ? $("#pg") : $("#pn")).focus();
}

/* ---------- overview ---------- */
function viewHome() {
  const st = S.state;
  if (!st.initialized) {
    $("#main").innerHTML = `${menuBtn()}<div class="head"><div><h2>Set up OpenSpec</h2><p>${esc(st.path)}</p></div></div>
      <div class="panel card"><h3>This project has no openspec/ folder yet</h3>
      <p>Setting it up creates <code>openspec/specs</code>, <code>openspec/changes</code> and a config file. Choosing a coding agent also adds its /opsx slash commands to the project.</p>
      <label for="tools">Coding agents to set up</label>
      <select id="tools"><option value="none">None, just the spec folders</option>
      <option value="codex">OpenAI Codex</option><option value="claude">Claude Code</option>
      <option value="cursor">Cursor</option><option value="github-copilot">GitHub Copilot</option>
      <option value="gemini">Gemini CLI</option><option value="opencode">OpenCode</option>
      <option value="agents">Generic AGENTS.md</option><option value="codex,claude">OpenAI Codex + Claude Code</option>
      <option value="all">All supported agents</option></select>
      <div class="row" style="margin-top:18px"><button class="btn primary" id="ib">Set up OpenSpec</button></div>
      <pre id="iout" class="muted pre"></pre></div>`;
    $("#ib").onclick = async e => {
      busy(e.target, true);
      try {
        const r = await api("/api/init", {method: "POST", body: {tools: $("#tools").value}});
        $("#iout").textContent = r.output; await refresh();
        if (S.state.initialized) { toast("OpenSpec is set up"); viewHome(); }
      } catch (err) { toast(err.message); } finally { busy(e.target, false); }
    };
    return;
  }
  $("#main").innerHTML = `${menuBtn()}<div class="head"><div><h2>${esc(st.project)}</h2>
    <p>${plural(st.changes.length, "active change")} · ${plural(st.specs.length, "spec")} · ${st.archive.length} archived</p></div>
    <div class="row"><button class="btn" id="va">Validate everything</button><button class="btn primary" onclick="go('new')">New change</button></div></div>
    <div id="vres"></div>
    ${st.changes.length ? `<div class="panel list" style="margin-bottom:18px">${st.changes.map(c => {
      const pct = c.totalTasks ? Math.round(100 * c.completedTasks / c.totalTasks) : 0;
      return `<button class="lrow" data-open="${esc(c.name)}"><b>${esc(c.name)}</b>
        <span class="muted">${c.totalTasks ? `${c.completedTasks}/${c.totalTasks} tasks` : "no tasks yet"}</span>
        <span class="meter"><i style="width:${pct}%"></i></span></button>`;
    }).join("")}</div>` : ""}
    <div class="panel card"><h3>How a change moves through OpenSpec</h3>
    <div class="doc flush"><ol>
      <li><b>Propose</b>: create a change and write <code>proposal.md</code>: why, and what changes.</li>
      <li><b>Specify</b>: add delta specs under <code>specs/</code> (ADDED, MODIFIED or REMOVED requirements) and a <code>design.md</code>.</li>
      <li><b>Plan</b>: break the work into a <code>tasks.md</code> checklist.</li>
      <li><b>Build</b>: implement it in your editor or coding agent (<code>/opsx:apply</code>), ticking tasks off here as you go.</li>
      <li><b>Archive</b>: merge the deltas into the main specs and move the change to the archive.</li>
    </ol></div></div>`;
  $("#va").onclick = e => runValidate(e.target, null);
  $("#main").querySelectorAll("[data-open]").forEach(b => b.onclick = () => go("change", b.dataset.open));
}

async function runValidate(btn, name) {
  busy(btn, true);
  try {
    const r = await api("/api/validate", {method: "POST", body: {name}});
    const box = $("#vres");
    if (!r.result) { box.innerHTML = `<div class="note err">${esc(r.error || "Validation failed to run")}</div>`; return; }
    const items = r.result.items || [];
    const bad = items.filter(i => !i.valid || (i.issues || []).some(x => !/^info$/i.test(x.level || "")));
    if (!items.length) box.innerHTML = `<div class="note warn">Nothing to validate yet.</div>`;
    else if (!bad.length) box.innerHTML = `<div class="note ok">All ${plural(items.length, "item")} passed strict validation.</div>`;
    else box.innerHTML = bad.map(validationNote).join("");
    bindValidationActions(box);
  } catch (err) { toast(err.message); } finally { busy(btn, false); }
}

const NO_DELTA = /at least one delta/i;

function validationNote(i) {
  if (i.type === "change" && (i.issues || []).some(x => NO_DELTA.test(x.message || ""))) {
    return `<div class="note warn"><b>${esc(i.id)}</b> has no spec deltas yet, so it can't pass validation.
      A spec delta lists the requirements this change adds or modifies, in <code>specs/&lt;capability&gt;/spec.md</code> inside the change.
      If the change touches no specs at all (a refactor, tooling or docs), mark it as such and OpenSpec will accept it.
      <div class="row" style="margin-top:12px"><button class="btn sm primary" data-addspec="${esc(i.id)}">Add a spec delta</button>
      ${runner() ? `<button class="btn sm agent" data-fix="${esc(i.id)}" data-kind="change">Let ${esc(runnerName())} write the spec deltas</button>` : ""}
      <button class="btn sm" data-skipspecs="${esc(i.id)}">This change touches no specs</button></div></div>`;
  }
  const fix = runner() ? `<div class="row" style="margin-top:10px"><button class="btn sm agent" data-fix="${esc(i.id)}" data-kind="${esc(i.type)}">Let ${esc(runnerName())} fix it</button></div>` : "";
  return `<div class="note ${i.valid ? "warn" : "err"}"><b>${esc(i.id)}</b> (${esc(i.type)}) ${i.valid ? "passed with warnings" : "failed"}
    <ul>${(i.issues || []).map(x => `<li>${esc(x.level ? x.level + ": " : "")}${esc(x.path ? x.path + ": " : "")}${esc(x.message || JSON.stringify(x))}</li>`).join("")}</ul>${fix}</div>`;
}

function bindValidationActions(box) {
  box.querySelectorAll("[data-fix]").forEach(b => b.onclick = () => runAgent(b.dataset.kind === "spec"
    ? {task: "fix-validation", spec: b.dataset.fix} : {task: "fix-validation", change: b.dataset.fix}));
  box.querySelectorAll("[data-addspec]").forEach(b => b.onclick = () => {
    if (!guard()) return;
    S.pendingStep = "specs";
    if (S.view.kind === "change" && S.view.name === b.dataset.addspec) viewChange(true); else go("change", b.dataset.addspec);
  });
  box.querySelectorAll("[data-skipspecs]").forEach(b => b.onclick = async e => {
    busy(e.target, true);
    try {
      await api(`/api/changes/${encodeURIComponent(b.dataset.skipspecs)}/skip-specs`, {method: "POST", body: {skip: true}});
      toast(`${b.dataset.skipspecs} marked as touching no specs`);
      await refresh();
      if (S.view.kind === "change" && S.view.name === b.dataset.skipspecs) await viewChange(true);
      runValidate(null, S.view.kind === "change" ? S.view.name : null);
    } catch (err) { toast(err.message); busy(e.target, false); }
  });
}

/* ---------- new change ---------- */
async function viewNew() {
  try { await getAgents(); } catch {}
  const ag = runner();
  $("#main").innerHTML = `${menuBtn()}<div class="head"><div><h2>New change</h2><p>A change holds everything about one piece of work: proposal, spec deltas, design and tasks.</p></div></div>
    <div class="panel card">
    <label for="cn">Change name</label><input type="text" id="cn" placeholder="add-dark-mode">
    <div class="hint">Lowercase words joined by hyphens, usually starting with a verb.</div>
    <label for="desc">What do you want? <span class="muted">(optional)</span></label>
    <textarea class="field" id="desc" placeholder="Describe the feature or fix in plain words: what it does, who it's for, anything it must or must not do. ${ag ? `${esc(ag.name)} turns this into the proposal, spec deltas, design and tasks.` : "Set up a coding agent (notebook step 1 and 2) to have it written for you."}"></textarea>
    <div class="row" style="margin-top:18px"><button class="btn primary" id="cb">Create change</button>
      ${ag ? `<button class="btn agent" id="cp">Create and let ${esc(ag.name)} write the plan</button>` : ""}
      <span class="hint" id="cbh"></span></div></div>`;
  $("#cn").focus();
  const create = async (plan) => {
    const btn = plan ? $("#cp") : $("#cb"), text = $("#desc").value.trim();
    if (plan && !text) { toast("Describe what you want first"); $("#desc").focus(); return; }
    busy(btn, true);
    try {
      const r = await api("/api/changes", {method: "POST", body: {name: $("#cn").value}});
      await refresh(); toast(`Created ${r.name}`); go("change", r.name);
      if (plan) runAgent({task: "plan", change: r.name, text});
    } catch (err) { toast(err.message); busy(btn, false); }
  };
  $("#cb").onclick = () => create(false);
  if ($("#cp")) $("#cp").onclick = () => create(true);
  $("#cn").onkeydown = e => { if (e.key === "Enter") create(false); };
}

/* ---------- change ---------- */
async function viewChange(keepFile) {
  const name = S.view.name;
  $("#main").innerHTML = `${menuBtn()}<p class="muted"><span class="spin"></span> Loading ${esc(name)}…</p>`;
  let c;
  try { [c] = await Promise.all([api(`/api/changes/${encodeURIComponent(name)}`), getAgents()]); }
  catch (err) { $("#main").innerHTML = `<div class="note err">${esc(err.message)}</div>`; return; }
  S.change = c;
  const arts = c.status?.artifacts || [];
  if (!keepFile || !c.files.includes(S.file)) S.file = c.files.find(f => f.endsWith("proposal.md")) || c.files.find(f => f.endsWith(".md")) || null;
  const t = c.tasks, pct = t.total ? Math.round(100 * t.done / t.total) : 0;
  const planned = arts.length > 0 && arts.every(a => a.status === "done" || a.status === "skipped");
  const built = t.total > 0 && t.done === t.total;
  const steps = [
    ...arts.map(a => ({id: a.id, kind: "art", status: a.status,
      line: a.status === "done" ? "Written" : a.status === "skipped" ? "Skipped" : a.status === "ready" ? "Ready to write" : "Needs " + (a.missingDeps || a.requires || []).join(", "),
      sub: a.status === "skipped" ? "Touches no specs" : STAGE_TEXT[a.id] || a.outputPath})),
    {id: "apply", kind: "wf", status: built ? "done" : planned ? "ready" : "blocked",
      line: t.total ? `${t.done}/${t.total} tasks done` : "Needs tasks", sub: "Build it in your agent"},
    {id: "verify", kind: "wf", status: built ? "ready" : "blocked", line: built ? "Ready" : "After apply", sub: "Check code matches specs"},
    {id: "archive", kind: "wf", status: built ? "ready" : "blocked", line: built ? "Ready" : "After verify", sub: "Merge specs, close change"},
  ];
  $("#main").innerHTML = `${menuBtn()}
    <div class="head"><div><h2>${esc(name)}</h2>
      <p>${planned ? "Planning complete" : "Planning in progress"} · ${t.total ? `${t.done} of ${t.total} tasks done (${pct}%)` : "no tasks yet"}</p></div>
      <div class="row"><button class="btn" id="sh">Show in console</button><button class="btn" id="vb">Validate</button><button class="btn primary" id="ab">Archive</button></div></div>
    <div id="vres"></div>
    ${arts.length ? `<div class="pipe" style="--n:${steps.length}">${steps.map(s => `<button class="stage ${esc(s.status)}${s.kind === "wf" ? " wf" : ""}" data-step="${esc(s.id)}">
      <i class="dot"></i><b>${esc(s.id)}</b><span>${esc(s.line)}</span><span>${esc(s.sub)}</span></button>`).join("")}</div>`
      : c.statusError ? `<div class="note warn">${esc(c.statusError)}</div>` : ""}
    <div id="next"></div>
    <div class="panel" id="filepanel"></div>`;
  $("#vb").onclick = e => runValidate(e.target, name);
  $("#ab").onclick = () => archiveChange(c);
  $("#sh").onclick = () => runConsole(`show ${name}`);
  $("#main").querySelectorAll("[data-step]").forEach(b => b.onclick = () => {
    const s = steps.find(x => x.id === b.dataset.step);
    if (s.kind === "art") return artifactClicked(arts.find(a => a.id === s.id));
    if (s.id === "archive" && built) return archiveChange(c);
    drawNext(c, s.id);
    $("#next").scrollIntoView({behavior: "smooth", block: "nearest"});
  });
  drawNext(c, null);
  renderFiles();
  if (S.pendingStep) { const id = S.pendingStep; S.pendingStep = null; artifactClicked(arts.find(a => a.id === id)); }
}

/* ---------- next step: what to run in your coding agent ---------- */
const WF_TEXT = {
  explore: "Think through an idea or problem before committing to a change",
  propose: "Create a change and write all its planning files in one go",
  new: "Create an empty change, then write its files one at a time",
  continue: "Write the next planning file of a change",
  ff: "Fast-forward: write all remaining planning files",
  apply: "Implement the tasks in the code, ticking them off",
  verify: "Check the implementation matches the specs and tasks",
  update: "Revise a change's planning files",
  sync: "Merge a change's spec deltas into the main specs without archiving",
  archive: "Finish a change: merge its specs and move it to the archive",
  "bulk-archive": "Archive several finished changes at once",
  onboard: "Guided walkthrough of OpenSpec on your codebase",
};

async function getAgents(force) { if (!S.agents || force) S.agents = await api("/api/agents"); return S.agents; }

function agentRows(wf, change) {
  const installed = S.agents?.runner?.installed || [];
  return (S.agents?.tools || []).filter(t => t.commands[wf]).map(t => {
    const call = t.commands[wf].call;
    const text = t.skills ? `Use the ${call} skill${change ? ` for the change ${change}` : ""}` : `${call}${change ? " " + change : ""}`;
    const here = installed.find(a => a.id === t.id);
    const run = here ? `<button class="btn sm agent" data-run="${esc(wf)}" data-agent="${esc(t.id)}" data-change="${esc(change || "")}" title="${esc(here.auth.ok ? "Runs here; output goes to the console" : here.auth.text)}">Run here</button>` : "";
    return `<div class="cmdrow${run ? " runnable" : ""}"><span class="who">${esc(t.name)}</span><code>${esc(text)}</code><span class="row">${run}<button class="btn sm" data-copy="${esc(text)}">Copy</button></span></div>`;
  }).join("");
}

function drawNext(c, focus) {
  const arts = c.status?.artifacts || [], t = c.tasks, ag = runner();
  const planned = arts.length > 0 && arts.every(a => a.status === "done" || a.status === "skipped");
  const anyDone = arts.some(a => a.status === "done");
  const built = t.total > 0 && t.done === t.total;
  let title, text, wfs, actions = "";
  const A = (attrs, label, primary) => `<button class="btn agent${primary ? " primary" : ""}" ${attrs}>${label}</button>`;
  if (focus === "apply" || (!focus && planned && !built)) {
    title = t.total ? `Next: build it (${t.done} of ${t.total} tasks done)` : "Next: build it";
    text = ag ? `${ag.name} implements tasks.md on this project's code and ticks the tasks off; you can also tick them in the file below.`
      : "OpenSpec doesn't write code itself. Your coding agent implements tasks.md on your project's code; tick tasks off here or let the agent do it.";
    wfs = ["apply"];
    if (ag) actions = A(`data-run="apply" data-change="${esc(c.name)}"`, `Build it with ${esc(ag.name)}`, true);
  } else if (focus === "verify" || focus === "archive" || (!focus && planned)) {
    title = "Next: verify, then archive";
    text = ag ? `All tasks are done. Have ${ag.name} check the code against the specs and tasks, press Validate, then Archive.`
      : "All tasks are done. Ask your agent to verify the code against the specs, press Validate here, then Archive.";
    wfs = ["verify", "archive"];
    if (ag) actions = A(`data-run="verify" data-change="${esc(c.name)}"`, `Verify with ${esc(ag.name)}`, true) + `<button class="btn" data-archive>Archive</button>`;
  } else {
    const next = arts.find(a => a.status === "ready");
    title = `Next: write ${next ? next.id : "the planning files"}`;
    if (ag) {
      text = next ? `${ag.name} writes ${next.id === "specs" ? "the spec deltas" : next.id + ".md"} from OpenSpec's template${anyDone ? " and the files written so far" : ""}; add a description if there's something it should know. Or write every remaining file in one go.`
        : `${ag.name} writes the planning files from OpenSpec's templates.`;
      actions = (next ? A(`data-write="${esc(next.id)}"`, `Write ${esc(next.id)} with ${esc(ag.name)}`, true) : "")
        + A(`data-plan`, anyDone ? "Write all remaining files" : "Write all planning files")
        + `<span class="hint">or click a stage above to write it yourself from the template</span>`;
    } else {
      text = "Click it in the row above to start from OpenSpec's template and write it here, or let your coding agent write it:";
    }
    wfs = anyDone ? ["continue", "ff", "update"] : ["propose"];
  }
  const tools = S.agents?.tools || [];
  const rows = ag ? "" : wfs.map(wf => { const r = agentRows(wf, c.name); return r ? `<div class="wfname">${esc(wf)} <span class="muted">· ${esc(WF_TEXT[wf] || "")}</span></div>${r}` : ""; }).join("");
  $("#next").innerHTML = `<div class="nextbox"><h3>${esc(title)}</h3><p class="muted">${esc(text)}</p>
    ${actions ? `<div class="row" style="margin-top:4px">${actions}</div>` : rows || `<p class="hint">${tools.length ? `Your agents don't have the ${wfs.join(" / ")} workflow yet. Turn on all workflows on the Agents page.`
      : "No coding agent is set up in this project. Add one (for example OpenAI Codex) on the Agents page."}
      <button class="btn sm" onclick="go('agents')">Open Agents page</button></p>`}</div>`;
  bindCopy($("#next"));
  const n = $("#next");
  n.querySelectorAll("[data-write]").forEach(b => b.onclick = () => artifactClicked(arts.find(a => a.id === b.dataset.write)));
  n.querySelector("[data-plan]") && (n.querySelector("[data-plan]").onclick = () => { if (!guard()) return; anyDone ? runAgent({task: "plan", change: c.name}) : planDialog(c); });
  n.querySelector("[data-archive]") && (n.querySelector("[data-archive]").onclick = () => archiveChange(c));
}

function planDialog(c) {
  const ag = runner();
  dialog(`<h3>Write the planning files with ${esc(ag.name)}</h3>
    <p class="muted">Proposal, spec deltas, design and tasks for <code>${esc(c.name)}</code>, from OpenSpec's templates, this project's code and your description.</p>
    <label for="desc">What do you want?</label><textarea class="field" id="desc" placeholder="Describe the feature or fix in plain words: what it does, who it's for, anything it must or must not do."></textarea>
    <div class="row" style="margin-top:20px"><button class="btn primary agent" id="go">Write them</button><button class="btn" id="cl" style="margin-left:auto">Cancel</button></div>`);
  $("#go").onclick = () => { const text = $("#desc").value.trim(); if (!text) { toast("Describe what you want first"); $("#desc").focus(); return; } $("#dlg").close(); runAgent({task: "plan", change: c.name, text}); };
  $("#desc").focus();
}

function bindCopy(root) {
  root.querySelectorAll("[data-copy]").forEach(b => b.onclick = async () => {
    try { await navigator.clipboard.writeText(b.dataset.copy); toast("Copied"); } catch { toast(b.dataset.copy); }
  });
  root.querySelectorAll("[data-run]").forEach(b => b.onclick = () => {
    if (!guard()) return;
    runAgent({task: "workflow", workflow: b.dataset.run, change: b.dataset.change || undefined, ...(b.dataset.agent ? {agent: b.dataset.agent} : {})});
  });
}

/* ---------- agents page ---------- */
async function viewAgents() {
  let a;
  try { a = await getAgents(true); } catch (err) { $("#main").innerHTML = `<div class="note err">${esc(err.message)}</div>`; return; }
  const allOn = a.allWorkflows.every(w => a.workflows.includes(w));
  $("#main").innerHTML = `${menuBtn()}<div class="head"><div><h2>Coding agents</h2>
    <p>OpenSpec doesn't run an AI itself. It installs its workflow into your coding agent as slash commands or skills. ${a.runner.installed.length ? "The agent's CLI is installed here, so this UI can run those workflows for you (see below); you can also run them in your editor or terminal." : "You run them there, in your editor or terminal, on this project's code."}</p></div></div>
    ${a.tools.length ? a.tools.map(tl => `<div class="panel card agentcard"><h3>${esc(tl.name)}
      <span class="authtag ${tl.auth.ok === true ? "ok" : tl.auth.ok === false ? "bad" : ""}">${esc(tl.auth.text)}</span></h3>
      <p class="muted">${tl.skills ? "Uses skills: mention the skill in your request, for example “Use the openspec-apply-change skill for add-login”."
        : "Uses slash commands: type the command in the agent's chat, followed by the change name."}</p>
      <div class="wftable">${Object.entries(tl.commands).map(([wf, x]) => `<div class="wfrow"><code>${esc(x.call)}</code><span>${esc(WF_TEXT[wf] || wf)}</span>
        <button class="btn sm" data-file="${esc(x.file)}">View file</button></div>`).join("")}</div></div>`).join("")
      : `<div class="note warn">No coding agent is set up in this project yet. Add one below.</div>`}
    <div class="panel card"><h3>Add a coding agent</h3>
      <p>Adds OpenSpec's workflow files for that agent to the project (the same as <code>openspec init --tools …</code>).</p>
      <div class="row"><select id="tl" style="max-width:320px">
        <option value="codex">OpenAI Codex</option><option value="claude">Claude Code</option><option value="cursor">Cursor</option>
        <option value="github-copilot">GitHub Copilot</option><option value="gemini">Gemini CLI</option><option value="opencode">OpenCode</option>
        <option value="agents">Generic AGENTS.md</option><option value="all">All supported agents</option></select>
        <button class="btn primary" id="addt">Add agent</button></div></div>
    <div class="panel card"><h3>Workflows</h3>
      <p>Which OpenSpec workflows your agents get. The core set covers most work; the full set adds step-by-step planning, verify and more. This setting is global on this machine.</p>
      <div class="wftable">${a.allWorkflows.map(wf => `<div class="wfrow"><span class="tick ${a.workflows.includes(wf) ? "on" : ""}">${a.workflows.includes(wf) ? "✓" : "–"}</span>
        <b>${esc(wf)}</b><span>${esc(WF_TEXT[wf])}</span></div>`).join("")}</div>
      <div class="row" style="margin-top:14px">${allOn ? `<button class="btn" id="core">Use the core set only</button>` : `<button class="btn primary" id="all">Turn on all ${a.allWorkflows.length} workflows</button>`}</div>
      <pre id="aout" class="pre muted"></pre></div>
    <div class="panel card"><h3>Running an agent from this UI</h3>
      ${a.runner.installed.length ? `<p>The <b>Run here</b>, <b>Write it with…</b> and <b>Let … fix it</b> buttons start this agent on the project and stream its output into the console. It ${a.runner.fullAccess ? "may edit files and run commands without asking" : "may only edit files in the project"} (<code>AGENT_FULL_ACCESS</code> in the notebook's step 1).</p>
      <div class="row"><label for="ra" style="margin:0">Agent</label><select id="ra" style="max-width:260px">${a.runner.installed.map(x => `<option value="${esc(x.id)}"${runner()?.id === x.id ? " selected" : ""}>${esc(x.name)} · ${esc(x.auth.text)}</option>`).join("")}</select>
        <label for="rm" style="margin:0">Model</label><input type="text" id="rm" style="max-width:220px" placeholder="agent's default" value="${esc(agentPrefs().model || "")}">
        <button class="btn" id="rs">Save</button></div>`
      : `<p>No agent CLI is installed on this machine, so the UI can only show you the commands to run elsewhere. Choose OpenAI Codex, Claude Code, Gemini CLI or OpenCode in the notebook's step 1 (with its API key), run step 2, then restart step 4.</p>`}
      <p class="muted" style="margin-top:12px">API keys are set in the notebook's step 1 (or Colab Secrets). Step 2 installs the agent's CLI and signs it in; the notebook's <b>Run your coding agent</b> cell does the same as the buttons here. You can also open the project on your computer (via GitHub or Google Drive) and use the agent there.</p></div>
    <div id="agfile"></div>`;
  if ($("#rs")) $("#rs").onclick = () => { saveAgentPrefs({agent: $("#ra").value, model: $("#rm").value.trim()}); toast(`The UI runs ${runnerName()}`); viewAgents(); };
  const run = async (btn, url, body, msg) => {
    busy(btn, true);
    try { const r = await api(url, {method: "POST", body}); $("#aout").textContent = r.output || ""; await getAgents(true); toast(msg); viewAgents(); }
    catch (err) { toast(err.message); } finally { busy(btn, false); }
  };
  $("#addt").onclick = e => run(e.target, "/api/agents/add", {tools: $("#tl").value}, "Agent added");
  $("#all") && ($("#all").onclick = e => run(e.target, "/api/agents/workflows", {preset: "all"}, "All workflows turned on"));
  $("#core") && ($("#core").onclick = e => run(e.target, "/api/agents/workflows", {preset: "core"}, "Core workflows only"));
  $("#main").querySelectorAll("[data-file]").forEach(b => b.onclick = () => {
    $("#agfile").innerHTML = `<div class="panel"><div id="agv"></div></div>`;
    S.editing = false; loadFile(b.dataset.file, $("#agv"), false);
    $("#agfile").scrollIntoView({behavior: "smooth"});
  });
}

function renderFiles() {
  const c = S.change, panel = $("#filepanel");
  const files = c.files.filter(f => !f.endsWith(".openspec.yaml"));
  if (!files.length) {
    panel.innerHTML = `<div class="card"><h3>No files yet</h3><p>Start with the proposal: click it in the row above to create it from OpenSpec's template.</p></div>`;
    return;
  }
  const base = c.dir + "/";
  panel.innerHTML = `<div class="tabs">${files.map(f =>
    `<button class="tab${f === S.file ? " on" : ""}" data-f="${esc(f)}">${esc(f.replace(base, ""))}</button>`).join("")}</div><div id="fileview"></div>`;
  panel.querySelectorAll("[data-f]").forEach(b => b.onclick = () => {
    if (!guard()) return;
    S.file = b.dataset.f; S.editing = false; S.dirty = false; renderFiles();
  });
  loadFile(S.file, $("#fileview"), true);
}

function artifactClicked(a) {
  if (!a) return;
  if (a.status === "blocked") { toast(`${a.id} comes after ${(a.missingDeps || []).join(" and ")}: write that first (click it in the row above)`); return; }
  if (a.status === "skipped") {
    dialog(`<h3>This change touches no specs</h3>
      <p class="muted"><code>skip_specs: true</code> is set in its <code>.openspec.yaml</code>, so OpenSpec accepts it without spec deltas. Turn that off if the change does add or modify requirements after all.</p>
      <div class="row" style="margin-top:20px"><button class="btn primary" id="unskip">Turn off and add a spec delta</button><button class="btn" id="cl" style="margin-left:auto">Keep it</button></div>`);
    $("#unskip").onclick = async e => {
      busy(e.target, true);
      try {
        await api(`/api/changes/${encodeURIComponent(S.change.name)}/skip-specs`, {method: "POST", body: {skip: false}});
        $("#dlg").close(); S.pendingStep = "specs"; await viewChange(true);
      } catch (err) { toast(err.message); busy(e.target, false); }
    };
    return;
  }
  const inChange = f => f.replace(S.change.dir + "/", "");
  const existing = S.change.files.filter(f => a.id === "specs" ? /^specs\/.+\.md$/.test(inChange(f)) : inChange(f) === a.outputPath);
  if (a.id !== "specs" && existing.length) { if (!guard()) return; S.file = existing[0]; S.editing = false; renderFiles(); return; }
  const specs = a.id === "specs", ag = runner();
  dialog(`<h3>${specs ? "Add a spec delta" : "Create " + esc(a.outputPath)}</h3>
    <p class="muted">${specs ? `One file per capability this change adds or modifies${existing.length ? `; this change has ${plural(existing.length, "spec file")}` : ""}.` : "Start from OpenSpec's template and write it yourself, or describe it and let your agent write it."}</p>
    ${specs ? `<label for="cap">Capability path${ag ? ` <span class="muted">(optional when ${esc(ag.name)} writes it)</span>` : ""}</label><input type="text" id="cap" list="caps" placeholder="user-auth">
      <datalist id="caps">${S.state.specs.map(s => `<option value="${esc(s.id)}">`).join("")}</datalist>
      <div class="hint">Use an existing spec's path to modify it, or a new path to add a capability.</div>` : ""}
    ${ag ? `<label for="desc">What should it say? ${a.id === "proposal" ? "" : `<span class="muted">(optional: ${esc(ag.name)} can derive it from the files so far)</span>`}</label><textarea class="field" id="desc" placeholder="${esc(specs ? "The requirements this change adds or modifies, in plain words. The agent turns them into ADDED / MODIFIED requirements with scenarios."
      : a.id === "tasks" ? "Anything the task list must cover or avoid; the agent derives the steps from the proposal, specs and design." : a.id === "design" ? "Constraints or preferences for how it's built; the agent derives the rest from the proposal and specs." : "The gist in plain words: what it does, who it's for, what it must or must not do.")}"></textarea>` : ""}
    <div class="row" style="margin-top:20px"><button class="btn${ag ? "" : " primary"}" id="tpl">Create from template</button>
      ${ag ? `<button class="btn primary agent" id="wr">Write it with ${esc(ag.name)}</button>` : ""}<button class="btn" id="cl" style="margin-left:auto">Cancel</button></div>`);
  $("#tpl").onclick = async e => {
    busy(e.target, true);
    try {
      const r = await api(`/api/changes/${encodeURIComponent(S.change.name)}/scaffold`, {method: "POST", body: {artifact: a.id, capability: $("#cap")?.value}});
      $("#dlg").close(); S.file = r.path; S.editing = true; await viewChange(true);
    } catch (err) { toast(err.message); } finally { busy(e.target, false); }
  };
  if ($("#wr")) $("#wr").onclick = () => {
    const text = $("#desc").value.trim();
    if (!text && a.id === "proposal") { toast("Describe what you want first"); $("#desc").focus(); return; }
    $("#dlg").close();
    runAgent({task: "write", change: S.change.name, artifact: a.id, capability: $("#cap")?.value, text});
  };
}

async function archiveChange(c) {
  const open = c.tasks.total - c.tasks.done;
  if (!confirm(`Archive ${c.name}?\n\nIts spec deltas are merged into openspec/specs and the change moves to the archive.${open > 0 ? `\n\n${plural(open, "task")} still open.` : ""}`)) return;
  try {
    const r = await api("/api/archive", {method: "POST", body: {name: c.name}});
    if (r.code !== 0) { $("#vres").innerHTML = `<div class="note err"><b>Archive failed</b><pre class="pre">${esc(r.output)}</pre></div>`; return; }
    toast(`Archived ${c.name}`); await refresh(); go("home");
  } catch (err) { toast(err.message); }
}

/* ---------- file viewer / editor ---------- */
async function loadFile(path, el, editable) {
  if (!path) { el.innerHTML = ""; return; }
  try {
    const r = await api(`/api/file?path=${encodeURIComponent(path)}`);
    S.content = r.content; S.file = path;
    drawFile(el, editable);
  } catch (err) { el.innerHTML = `<div class="card note err">${esc(err.message)}</div>`; }
}

function drawFile(el, editable) {
  const isMd = S.file.endsWith(".md");
  const mode = isMd ? (S.mode || "preview") : "source";
  const modes = isMd ? `<div class="seg">${[["preview", "Preview"], ["source", "Markdown"], ["split", "Split"]].map(([m, l]) =>
    `<button class="${mode === m ? "on" : ""}" data-mode="${m}">${l}</button>`).join("")}</div>` : "";
  const preview = text => isMd ? `<div class="doc">${md(text)}</div>` : `<pre class="src">${esc(text)}</pre>`;
  const source = text => `<pre class="src">${esc(text)}</pre>`;
  let body;
  if (S.editing) {
    body = `<div class="${isMd ? "split" : ""}"><textarea class="editor" id="ta" spellcheck="false"></textarea>${isMd ? `<div class="pane" id="live"></div>` : ""}</div>`;
  } else if (mode === "split") {
    body = `<div class="split"><div class="pane">${source(S.content)}</div><div class="pane">${preview(S.content)}</div></div>`;
  } else {
    body = mode === "source" ? source(S.content) : preview(S.content);
  }
  el.innerHTML = `<div class="toolbar"><code>${esc(S.file)}</code><div class="row">
    ${S.editing ? "" : modes}
    ${editable ? (S.editing ? `<button class="btn" id="cx">Cancel</button><button class="btn primary" id="sv">Save</button>`
      : `<button class="btn danger" id="dl">Delete</button><button class="btn" id="ed">Edit</button>`) : ""}</div></div>${body}`;
  const q = s => el.querySelector(s);
  q(".seg")?.querySelectorAll("[data-mode]").forEach(b => b.onclick = () => { S.mode = b.dataset.mode; drawFile(el, editable); });
  if (S.editing) {
    const ta = q("#ta"), live = q("#live");
    const sync = () => { if (live) live.innerHTML = `<div class="doc">${md(ta.value)}</div>`; };
    ta.value = S.content; sync();
    ta.oninput = () => { S.dirty = true; sync(); };
    ta.focus();
  }
  if (q("#ed")) q("#ed").onclick = () => { S.editing = true; drawFile(el, editable); };
  if (q("#cx")) q("#cx").onclick = () => { if (!guard()) return; S.editing = false; S.dirty = false; drawFile(el, editable); };
  if (q("#sv")) q("#sv").onclick = () => saveFile(q("#ta").value);
  if (q("#dl")) q("#dl").onclick = async () => {
    if (!confirm(`Delete ${S.file}?`)) return;
    await api(`/api/file?path=${encodeURIComponent(S.file)}`, {method: "DELETE"});
    toast("Deleted"); S.file = null; await refresh(); route();
  };
  if (editable && !S.editing) el.querySelectorAll(".doc input[type=checkbox]").forEach((cb, i) => {
    cb.disabled = false;
    cb.onchange = () => {
      let n = -1;
      const next = S.content.split("\n").map(line =>
        /^\s*[-*] \[[ xX]\]/.test(line) && ++n === i ? line.replace(/\[[ xX]\]/, cb.checked ? "[x]" : "[ ]") : line).join("\n");
      saveFile(next, true);
    };
  });
}

async function saveFile(content, quiet) {
  try {
    await api("/api/file", {method: "PUT", body: {path: S.file, content}});
    S.content = content; S.editing = false; S.dirty = false;
    if (!quiet) toast("Saved");
    await refresh();
    S.view.kind === "change" ? viewChange(true) : route();
  } catch (err) { toast(err.message); }
}

/* ---------- specs & archive ---------- */
function viewSpec() {
  const s = S.state.specs.find(x => x.id === S.view.name);
  if (!s) return viewHome();
  $("#main").innerHTML = `${menuBtn()}<div class="head"><div><h2>${esc(s.id)}</h2>
    <p>Main spec · ${plural(s.requirements, "requirement")}. To change it, create a change with a delta for this path.</p></div>
    <div class="row"><button class="btn" id="vb">Validate</button></div></div><div id="vres"></div><div class="panel" id="fv"></div>`;
  $("#vb").onclick = e => runValidate(e.target, s.id);
  S.editing = false; loadFile(s.path, $("#fv"), true);
}

async function viewArchived() {
  const r = await api(`/api/archive/${encodeURIComponent(S.view.name)}`);
  $("#main").innerHTML = `${menuBtn()}<div class="head"><div><h2>${esc(r.name)}</h2><p>Archived change, read only</p></div></div>
    <div class="panel"><div class="tabs">${r.files.map((f, i) => `<button class="tab${i ? "" : " on"}" data-f="${esc(f)}">${esc(f.split("/").slice(4).join("/"))}</button>`).join("")}</div><div id="fv"></div></div>`;
  const tabs = $("#main").querySelectorAll("[data-f]");
  tabs.forEach(b => b.onclick = () => { tabs.forEach(t => t.classList.remove("on")); b.classList.add("on"); loadFile(b.dataset.f, $("#fv"), false); });
  if (r.files[0]) loadFile(r.files[0], $("#fv"), false);
}

/* ---------- console ---------- */
function commandSuggestions() {
  const st = S.state, out = ["list", "list --specs", "validate --all --strict", "doctor", "schemas", "config list", "--help"];
  for (const c of st.changes) out.push(`show ${c.name}`, `status --change ${c.name}`, `validate ${c.name} --strict`,
    `show ${c.name} --json --deltas-only`, `instructions apply --change ${c.name}`, `archive ${c.name} --yes`);
  for (const s of st.specs) out.push(`show ${s.id} --type spec`, `validate ${s.id} --type spec --strict`);
  return out;
}

function explain(cmd, output) {
  if (!/Unknown item|not found|No change named/i.test(output)) return output;
  const st = S.state;
  return output + `\n\nNames that exist in this project:\n  changes: ${st.changes.map(c => c.name).join(", ") || "(none yet: create one with New change)"}` +
    `\n  specs:   ${st.specs.map(s => s.id).join(", ") || "(none yet: specs appear after you archive a change)"}`;
}

/* The console lives in a dock on the right of every page. Its DOM is built once (renderDock) so the
   output, scroll position and half-typed command survive navigation; only the hints are refreshed. */
function dockOpen(on) {
  S.dock = !!on;
  $(".app")?.classList.toggle("dock-closed", !S.dock);
  $("#dockToggle")?.classList.toggle("on", S.dock);
  try { localStorage.setItem("os.dock", S.dock ? "1" : "0"); } catch {}
  if (S.dock && window.innerWidth > 820) $("#cmd")?.focus();
}

function renderDock() {
  $("#dock").innerHTML = `<div class="dock-grip" id="grip" title="Drag to resize"></div>
    <div class="dock-head"><b>Console</b><span class="muted">openspec …</span>
      <button class="btn sm agent" id="ask" title="Give your coding agent an instruction; its output streams here">Ask agent</button>
      <button class="btn sm" id="clr" title="Clear output">Clear</button>
      <button class="btn sm" id="dockClose" title="Hide the console">×</button></div>
    <div class="chips" id="chips"></div>
    <div class="console" id="con"></div>
    <form class="cmdline" id="cf"><input type="text" id="cmd" list="cmds" autocomplete="off" spellcheck="false">
      <datalist id="cmds"></datalist><button class="btn primary">Run</button></form>
    <p class="cmdhelp" id="cmdhelp"></p>`;
  updateDockHints(); drawConsole();
  $("#cf").onsubmit = e => { e.preventDefault(); const v = $("#cmd").value; $("#cmd").value = ""; runConsole(v); };
  $("#clr").onclick = () => { S.log = S.log.filter(l => l.running); drawConsole(); };
  $("#dockClose").onclick = () => dockOpen(false);
  $("#ask").onclick = () => {
    const ag = runner();
    if (!ag) { toast("No coding agent CLI is installed here: choose one in the notebook's step 1 and run step 2"); return; }
    const change = S.view.kind === "change" ? S.view.name : "";
    dialog(`<h3>Ask ${esc(ag.name)}</h3><p class="muted">Runs in the project folder with your OpenSpec files and code. ${change ? `Mention <code>${esc(change)}</code> if it's about that change.` : ""}</p>
      <label for="desc">Instruction</label><textarea class="field" id="desc" placeholder="${esc(change ? `e.g. Review the design of ${change} and list risks in design.md` : "e.g. Summarise what this project does and what its specs cover")}"></textarea>
      <div class="row" style="margin-top:20px"><button class="btn primary agent" id="go">Run</button><button class="btn" id="cl" style="margin-left:auto">Cancel</button></div>`);
    $("#go").onclick = () => { const text = $("#desc").value.trim(); if (!text) return; $("#dlg").close(); runAgent({task: "custom", text}); };
    $("#desc").focus();
  };
  const grip = $("#grip");
  grip.onpointerdown = e => {
    e.preventDefault(); grip.setPointerCapture(e.pointerId); document.body.style.cursor = "col-resize";
    grip.onpointermove = ev => {
      const w = Math.round(Math.min(Math.max(window.innerWidth - ev.clientX, 280), window.innerWidth * 0.6));
      document.documentElement.style.setProperty("--dock", w + "px");
    };
    grip.onpointerup = grip.onpointercancel = () => {
      grip.onpointermove = null; document.body.style.cursor = "";
      try { localStorage.setItem("os.dockw", document.documentElement.style.getPropertyValue("--dock")); } catch {}
    };
  };
}

function updateDockHints() {
  const st = S.state, first = st.changes[0]?.name;
  if (!$("#chips")) return;
  const chips = ["list", "validate --all --strict", ...(first ? [`show ${first}`, `status --change ${first}`] : ["list --specs"]), "--help"];
  $("#chips").innerHTML = chips.map(q => `<button class="chip" data-q="${esc(q)}">${esc(q)}</button>`).join("");
  $("#chips").querySelectorAll("[data-q]").forEach(b => b.onclick = () => runConsole(b.dataset.q));
  $("#cmds").innerHTML = commandSuggestions().map(c => `<option value="${esc(c)}">`).join("");
  $("#cmd").placeholder = first ? `show ${first}` : "list --specs";
  $("#cmdhelp").innerHTML = `Changes: ${st.changes.map(c => `<code>${esc(c.name)}</code>`).join(" ") || "none yet"} · Specs: ${st.specs.map(s => `<code>${esc(s.id)}</code>`).join(" ") || "none yet"}`;
}

function drawConsole() {
  const c = $("#con");
  if (!c) return;
  const stick = c.scrollHeight - c.scrollTop - c.clientHeight < 40;
  c.innerHTML = S.log.map(l => l.kind === "agent"
    ? `<span class="a">▶ ${esc(l.cmd)}${l.running ? ` <button class="btn sm danger" data-stop="${esc(l.job || "")}">Stop</button>` : ""}</span>\n<span class="${l.code ? "e" : ""}">${esc(l.out)}</span>${l.running ? '<span class="spin"></span>' : ""}\n`
    : `<span class="c">$ openspec ${esc(l.cmd)}</span>\n<span class="${l.code ? "e" : ""}">${esc(l.out)}</span>\n`).join("\n")
    || `<span class="c">Output appears here. Run any non-interactive openspec command in the project folder (start typing for suggestions), or press Ask agent.</span>`;
  c.querySelectorAll("[data-stop]").forEach(b => b.onclick = async () => { b.disabled = true; try { await api(`/api/agent/jobs/${b.dataset.stop}/stop`, {method: "POST"}); } catch (err) { toast(err.message); } });
  if (stick) c.scrollTop = c.scrollHeight;
}

/* ---------- coding agent runs: started from the UI, streamed into the console ---------- */
function agentPrefs() { try { return JSON.parse(localStorage.getItem("os.agent") || "{}"); } catch { return {}; } }
function saveAgentPrefs(p) { try { localStorage.setItem("os.agent", JSON.stringify(p)); } catch {} }
function runner() {
  const r = S.agents?.runner;
  if (!r || !r.installed.length) return null;
  const want = agentPrefs().agent;
  return r.installed.find(a => a.id === want) || r.installed.find(a => a.id === r.default) || r.installed[0];
}
function runnerName() { return runner()?.name || "the agent"; }

async function runAgent(body) {
  const ag = body.agent ? (S.agents?.runner?.installed || []).find(a => a.id === body.agent) : runner();
  if (!ag) { toast("No coding agent CLI is installed here: choose one in the notebook's step 1 and run step 2"); return; }
  if (ag.auth.ok === false && !(ag.id === "codex" && S.agents?.runner?.keys?.includes("OPENAI_API_KEY"))) {
    toast(`${ag.name} isn't signed in: ${ag.auth.text}. Set its API key in the notebook's step 1, run step 2, then step 4.`); return;
  }
  if (S.log.some(l => l.running)) { toast("An agent is already running; stop it or wait for it to finish"); dockOpen(true); return; }
  dockOpen(true);
  const entry = {kind: "agent", cmd: `${ag.name} · starting…`, out: "", code: null, running: true};
  S.log.push(entry); drawConsole();
  try {
    const r = await api("/api/agent/run", {method: "POST", body: {agent: ag.id, model: agentPrefs().model || undefined, ...body}});
    entry.job = r.id; entry.cmd = r.title; drawConsole();
    pollJob(entry);
  } catch (err) { entry.cmd = `${ag.name} · could not start`; entry.out = err.message; entry.code = 1; entry.running = false; drawConsole(); }
}

async function pollJob(entry) {
  let seen = 0, failures = 0;
  while (entry.running) {
    try {
      const r = await api(`/api/agent/jobs/${entry.job}?offset=${seen}`);
      failures = 0;
      if (r.output) { entry.out += r.output; seen = r.length; drawConsole(); }
      if (r.status !== "running" && r.status !== "stopping") {
        entry.running = false; entry.code = r.status === "stopped" ? 130 : r.code;
        entry.out += `\n--- ${r.status === "stopped" ? "stopped" : `exit code ${r.code}`} after ${r.seconds}s · files changed in the project:\n${r.changed || "(none)"}\n`;
        drawConsole();
        await Promise.all([refresh(), getAgents(true)]).catch(() => {});
        if (!S.dirty) await (S.view.kind === "change" ? viewChange(true) : route());
        entry.settled = true;
        toast(r.status === "stopped" ? "Agent stopped" : r.code === 0 ? "Agent finished" : `Agent exited with code ${r.code}`);
        return;
      }
    } catch (err) {
      if (++failures > 5) { entry.running = false; entry.code = 1; entry.out += `\n${err.message}`; drawConsole(); return; }
    }
    await new Promise(r => setTimeout(r, 1000));
  }
}

async function runConsole(cmd) {
  cmd = String(cmd || "").trim().replace(/^openspec\s+/, "");
  if (!cmd) return;
  dockOpen(true);
  S.log.push({cmd, out: "…", code: 0}); drawConsole();
  try { const r = await api("/api/run", {method: "POST", body: {cmd}}); S.log[S.log.length - 1] = {cmd, out: explain(cmd, r.output), code: r.code}; }
  catch (err) { S.log[S.log.length - 1] = {cmd, out: err.message, code: 1}; }
  drawConsole(); refresh();
}

/* ---------- start ---------- */
window.addEventListener("beforeunload", e => { if (S.dirty) { e.preventDefault(); e.returnValue = ""; } });
boot();


Writing /content/openspec-studio/static/app.js


## 4 · Start

In [8]:
import re, socket, time, shutil
from IPython.display import display, HTML

if "codex" in AGENTS and shutil.which("codex"):
    if not sign_in_codex():
        print("⚠ OpenAI Codex is not signed in: set OPENAI_API_KEY in step 1 and run step 2, or the agent buttons in the UI won't work.")

LOG_DIR = "/content/openspec-logs"
os.makedirs(LOG_DIR, exist_ok=True)
subprocess.run("pkill -f openspec-studio/server.py; pkill -f 'cloudflared tunnel'", shell=True)
time.sleep(1)

def start(cmd, name, env=None):
    log = open(f"{LOG_DIR}/{name}.log", "w")
    return subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT,
                            env={**os.environ, **(env or {})}, start_new_session=True)

server = start(f"python3 {APP_DIR}/server.py", "server",
               env={"PROJECTS_DIR": PROJECTS_DIR, "AGENT_TOOLS": AGENT_TOOLS, "APP_PASSWORD": UI_PASSWORD, "PORT": str(PORT),
                    "AGENT_FULL_ACCESS": "1" if AGENT_FULL_ACCESS else "0", "DEFAULT_AGENT": AGENTS[0] if AGENTS else ""})
for _ in range(40):
    with socket.socket() as s:
        if s.connect_ex(("127.0.0.1", PORT)) == 0:
            break
    time.sleep(0.5)
else:
    raise RuntimeError(open(f"{LOG_DIR}/server.log").read()[-1500:])

tunnel = start(f"cloudflared tunnel --no-autoupdate --url http://127.0.0.1:{PORT}", "cloudflared")
URL = None
for _ in range(90):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(f"{LOG_DIR}/cloudflared.log").read())
    if m:
        URL = m.group(0)
        break
    time.sleep(1)
if not URL:
    raise RuntimeError(f"No tunnel link yet; see {LOG_DIR}/cloudflared.log and run this cell again")

time.sleep(4)
display(HTML(f"""<div style="font:15px/1.7 sans-serif;padding:12px 16px;border:1px solid #9aa;border-radius:10px;max-width:600px">
<b>OpenSpec Studio is running</b><br>
Open: <a href="{URL}" target="_blank"><b>{URL}</b></a><br>
Password: <code>{UI_PASSWORD}</code><br>
Then create or open a project under <b>Projects</b> in the sidebar.<br>
<span style="color:#667">If the page can't be reached yet, wait 20–30 seconds and reload.</span></div>"""))

## 5 · Run your coding agent


In [9]:
AGENT = "codex"  #@param ["codex", "claude", "gemini", "opencode"]
WORKFLOW = "apply"  #@param ["propose", "explore", "apply", "verify", "update", "continue", "ff", "sync", "archive", "onboard", "custom prompt"]
CHANGE = ""  #@param {type:"string"}
#@markdown ↑ the change name (for **propose**: the change name or your idea in a sentence)
EXTRA = ""  #@param {type:"string"}
#@markdown ↑ extra instructions, or the whole prompt when WORKFLOW is "custom prompt"
AGENT_MODEL = ""  #@param {type:"string"}
#@markdown ↑ optional model id; empty = the agent's default
PROJECT = ""  #@param {type:"string"}
#@markdown ↑ empty = the project currently open in the UI

import json, shlex
from IPython.display import display, HTML

# where each agent keeps OpenSpec's instructions for a workflow (written by openspec init)
SKILL = {"propose": "openspec-propose", "explore": "openspec-explore", "apply": "openspec-apply-change",
         "verify": "openspec-verify-change", "update": "openspec-update-change", "continue": "openspec-continue-change",
         "ff": "openspec-ff-change", "sync": "openspec-sync-specs", "archive": "openspec-archive-change",
         "onboard": "openspec-onboard"}
CHANGE_BOUND = ("apply", "verify", "update", "continue", "ff", "sync", "archive")
# what to do when the agent has no skill file for the workflow (the core profile installs only six)
WF_FALLBACK = {
    "continue": "write the next missing planning artifact: run `openspec status --change {q} --json` to find it and "
                "`openspec instructions <artifact> --change {q} --json` for its template and rules, read the artifacts already written and base it on them",
    "ff": "write every planning artifact the change is still missing, in dependency order (proposal, spec deltas, design, tasks): use "
          "`openspec status --change {q} --json` and `openspec instructions <artifact> --change {q} --json`, base each on the artifacts already written, "
          "then run `openspec validate {q} --strict` and fix anything it reports",
    "verify": "check that the implementation matches the change: read openspec/changes/{c}/ (proposal, specs, design, tasks), inspect the code and tests, "
              "list what is missing, different or untested, and correct the task ticks in tasks.md to reflect reality; do not archive",
}
NON_INTERACTIVE = ("You are running non-interactively and nobody can answer questions: do not ask any, make reasonable assumptions "
                   "and note them in what you write. Work only inside this project; it already uses OpenSpec, never run openspec init.")

def instructions_file(agent, wf):
    candidates = {
        "codex": [f".agents/skills/{SKILL.get(wf)}/SKILL.md"],
        "claude": [f".claude/commands/opsx/{wf}.md", f".claude/skills/{SKILL.get(wf)}/SKILL.md"],
        "gemini": [f".gemini/commands/opsx/{wf}.toml"],
        "opencode": [f".opencode/commands/opsx-{wf}.md"],
    }[agent] + [f".agents/skills/{SKILL.get(wf)}/SKILL.md"]
    return next((c for c in candidates if os.path.exists(os.path.join(PROJECT_DIR, c))), None)

def active_changes():
    d = os.path.join(PROJECT_DIR, "openspec", "changes")
    return sorted(n for n in os.listdir(d) if n != "archive" and os.path.isdir(os.path.join(d, n))) if os.path.isdir(d) else []

def stop(msg):     # a clear box instead of a traceback; nothing runs after it
    display(HTML(f'<div style="font:15px/1.6 sans-serif;padding:12px 16px;border-left:4px solid #e5ae55;background:#fff8e6;color:#5a4300;max-width:700px">{msg}</div>'))

def current_project():
    """The project open in the UI: the server writes it to PROJECTS_DIR/.openspec-studio.json."""
    try:
        return json.load(open(os.path.join(PROJECTS_DIR, ".openspec-studio.json"))).get("current") or ""
    except (OSError, ValueError):
        return ""

def run_agent():
    global PROJECT_DIR
    name = PROJECT.strip() or current_project()
    if not name or not os.path.isdir(os.path.join(PROJECTS_DIR, name)):
        return stop("No project is open. Create or open one under <b>Projects</b> in the UI (step 4), or set <b>PROJECT</b> above to a folder name in "
                    f"<code>{PROJECTS_DIR}</code>.")
    PROJECT_DIR = os.path.join(PROJECTS_DIR, name)
    print("Project:", PROJECT_DIR)
    if AGENT == "codex" and not sign_in_codex():
        return stop("<b>OpenAI Codex is not signed in</b>, so it would fail with <code>401 Unauthorized</code>. Set <code>OPENAI_API_KEY</code> in step 1 (or Colab Secrets), run step 2, then run this cell again.")
    if AGENT == "claude" and not os.environ.get("ANTHROPIC_API_KEY"):
        return stop("<b>Claude Code needs <code>ANTHROPIC_API_KEY</code></b>: set it in step 1 and run step 2.")
    if AGENT == "gemini" and not os.environ.get("GEMINI_API_KEY"):
        return stop("<b>Gemini CLI needs <code>GEMINI_API_KEY</code></b>: set it in step 1 and run step 2.")

    change = CHANGE.strip()
    if WORKFLOW in CHANGE_BOUND and not change:          # pick the change when there is only one; otherwise ask
        names = active_changes()
        if len(names) == 1:
            change = names[0]
            print(f"CHANGE is empty; using the only active change: {change}")
        else:
            return stop(f"The <b>{WORKFLOW}</b> workflow works on one change. Set <b>CHANGE</b> above to "
                        + (", ".join(f"<code>{n}</code>" for n in names) if names else "a change name (this project has none yet: create one in the UI)")
                        + " and run this cell again. Tip: the UI does this for you with <b>Build it with…</b> / <b>Verify with…</b> on the change's page.")

    if WORKFLOW == "custom prompt":
        prompt = EXTRA
        if not prompt.strip():
            return stop("Type the prompt in <b>EXTRA</b> when WORKFLOW is <i>custom prompt</i>.")
    else:
        target = f' for the change "{change}"' if change else ""
        file = instructions_file(AGENT, WORKFLOW)
        if file:
            prompt = f'Follow the instructions in {file} to run the OpenSpec "{WORKFLOW}" workflow{target}. {EXTRA}'.strip()
        elif WORKFLOW in WF_FALLBACK and change:
            quoted = f'"{change}"'
            prompt = f'Run the OpenSpec "{WORKFLOW}" workflow{target}: {WF_FALLBACK[WORKFLOW].format(q=quoted, c=change)}. {EXTRA}'.strip()
        else:
            return stop(f"No OpenSpec <b>{WORKFLOW}</b> instructions found for {AGENT}. Add the agent on the UI's Agents page "
                        "(or turn on all workflows there), then run this cell again.")
        prompt += "\n\n" + NON_INTERACTIVE

    model = ["-m", AGENT_MODEL] if AGENT_MODEL and AGENT != "claude" else (["--model", AGENT_MODEL] if AGENT_MODEL else [])
    env = dict(os.environ)
    if AGENT == "codex":
        cmd = ["codex", "exec", "--skip-git-repo-check", "-C", PROJECT_DIR, *model,
               *(["--dangerously-bypass-approvals-and-sandbox"] if AGENT_FULL_ACCESS else ["--sandbox", "workspace-write"]), prompt]
    elif AGENT == "claude":
        env["IS_SANDBOX"] = "1"   # Colab runs as root; Claude Code only allows full access as root inside a sandbox
        cmd = ["claude", "-p", prompt, *model,
               *(["--dangerously-skip-permissions"] if AGENT_FULL_ACCESS else ["--permission-mode", "acceptEdits"])]
    elif AGENT == "gemini":
        cmd = ["gemini", "-p", prompt, *model, *(["--yolo"] if AGENT_FULL_ACCESS else ["--approval-mode", "auto_edit"])]
    else:
        cmd = ["opencode", "run", *model, prompt]

    short = prompt[:120] + ("…" if len(prompt) > 120 else "")
    print("$", " ".join(shlex.quote(short if c == prompt else c) for c in cmd), "\n")
    proc = subprocess.Popen(cmd, cwd=PROJECT_DIR, env=env, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in proc.stdout:          # stream the agent's output as it works
            print(line, end="")
        proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        print("\nStopped the agent.")
    print(f"\n--- exit code {proc.returncode} · files changed in the project:")
    print(sh(f"git -C '{PROJECT_DIR}' status --short") or "(none)")

run_agent()


## 6 · Keep alive

In [ ]:
try:
    while server.poll() is None and tunnel.poll() is None:
        print(f"[{time.strftime('%H:%M:%S')}] running · {URL}", end="\r")
        time.sleep(30)
    print(f"\nSomething stopped. Check {LOG_DIR} and run step 5 again.")
except KeyboardInterrupt:
    print("\nStopped waiting; the UI is still running.")

## 7 · Stop

In [ ]:
for p in (server, tunnel):
    p.terminate()
print("Stopped. The link no longer works.")